# NMR representations in Cox models


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index

try:
    from sksurv.metrics import concordance_index_ipcw
    from sksurv.util import Surv
except ImportError as error:
    raise ImportError(
        "scikit-survival is required for Uno IPCW C-index and Coxnet. "
        "Install it in the server environment with: uv pip install scikit-survival"
    ) from error

try:
    from dcurves import dca as dcurves_dca
except ImportError as error:
    raise ImportError(
        "dcurves is required for standard survival DCA. "
        "Install it in the server environment with: uv pip install 'dcurves>=1.1.7'"
    ) from error

CNS = {
    "blue": "#3C5488",
    "red": "#E64B35",
    "green": "#00A087",
    "teal": "#4DBBD5",
    "orange": "#F39B7F",
    "purple": "#8491B4",
    "gray": "#6E6E6E",
    "light_gray": "#D9D9D9",
    "black": "#222222",
}

plt.rcParams.update({
    "figure.dpi": 170,
    "savefig.dpi": 600,
    "font.family": "DejaVu Sans",
    "font.size": 13.0,
    "axes.titlesize": 15.0,
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.0,
    "ytick.labelsize": 12.0,
    "legend.fontsize": 11.0,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

HORIZONS = [5.0, 10.0]
NRI_CUTPOINTS = [
    float(value)
    for value in os.environ.get("CHD_NRI_CUTPOINTS", "0.05,0.10,0.20").split(",")
]
NRI_CUTPOINT_SOURCE = os.environ.get("CHD_NRI_CUTPOINT_SOURCE", "").strip()
if sorted(NRI_CUTPOINTS) != NRI_CUTPOINTS or any(
    value <= 0 or value >= 1 for value in NRI_CUTPOINTS
):
    raise ValueError("CHD_NRI_CUTPOINTS must be strictly increasing probabilities in (0, 1).")
CATEGORICAL_NRI_ENABLED = bool(NRI_CUTPOINT_SOURCE)
if not CATEGORICAL_NRI_ENABLED:
    warnings.warn("CHD_NRI_CUTPOINT_SOURCE is unset: categorical NRI will be omitted; continuous NRI and IDI remain active.")
BOOTSTRAP_N = int(os.environ.get("CHD_BOOTSTRAP_N", "1000"))
BOOTSTRAP_SEED = int(os.environ.get("CHD_BOOTSTRAP_SEED", "2026"))
IBS_GRID_N = int(os.environ.get("CHD_IBS_GRID_N", "51"))
DCA_THRESHOLDS = np.arange(0.01, 0.201, 0.005)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return str(path)

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def save_figure(fig, output_dir, stem):
    ensure_dir(output_dir)
    for extension in ["png", "pdf", "svg"]:
        fig.savefig(
            os.path.join(output_dir, f"{stem}.{extension}"),
            dpi=600,
            bbox_inches="tight",
        )
    print("Saved figure:", os.path.join(output_dir, stem + ".png"))

def quantile_ci(values, lower=0.025, upper=0.975):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    return float(np.quantile(values, lower)), float(np.quantile(values, upper))

def make_surv(frame):
    return Surv.from_arrays(
        event=frame["Event"].astype(bool).to_numpy(),
        time=frame["Time"].astype(float).to_numpy(),
    )

def eid_digest(frame):
    if "Eid" not in frame.columns:
        return "no_eid"
    values = pd.util.hash_pandas_object(frame["Eid"], index=False).to_numpy()
    return hashlib.sha256(values.tobytes()).hexdigest()

def harrell_c(frame, risk):
    return float(
        concordance_index(
            frame["Time"].astype(float).to_numpy(),
            -np.asarray(risk, dtype=float).reshape(-1),
            frame["Event"].astype(int).to_numpy(),
        )
    )

def uno_c(train_frame, eval_frame, risk, tau):
    if float(tau) <= 0:
        raise ValueError("Uno C tau must be positive.")
    estimate = concordance_index_ipcw(
        make_surv(train_frame),
        make_surv(eval_frame),
        np.asarray(risk, dtype=float).reshape(-1),
        tau=float(tau),
    )[0]
    return float(estimate)

def paired_bootstrap_uno(
    train_frame,
    eval_frame,
    risk_new,
    risk_reference,
    tau,
    bootstrap_n=BOOTSTRAP_N,
    seed=BOOTSTRAP_SEED,
):
    risk_new = np.asarray(risk_new, dtype=float).reshape(-1)
    risk_reference = np.asarray(risk_reference, dtype=float).reshape(-1)
    original_new = uno_c(train_frame, eval_frame, risk_new, tau)
    original_reference = uno_c(train_frame, eval_frame, risk_reference, tau)
    original_delta = original_new - original_reference

    rng = np.random.default_rng(seed)
    n = len(eval_frame)
    values = []
    for _ in range(int(bootstrap_n)):
        idx = rng.integers(0, n, size=n)
        boot_frame = eval_frame.iloc[idx].reset_index(drop=True)
        if boot_frame["Event"].nunique() < 2:
            continue
        try:
            values.append(
                uno_c(train_frame, boot_frame, risk_new[idx], tau)
                - uno_c(train_frame, boot_frame, risk_reference[idx], tau)
            )
        except Exception:
            continue
    lower, upper = quantile_ci(values)
    return {
        "Uno_C_New": original_new,
        "Uno_C_Reference": original_reference,
        "Delta_Uno_C": original_delta,
        "Delta_Uno_C_Lower95": lower,
        "Delta_Uno_C_Upper95": upper,
        "Bootstrap_N_Used": int(len(values)),
    }

def paired_bootstrap_harrell(
    eval_frame,
    risk_new,
    risk_reference,
    bootstrap_n=BOOTSTRAP_N,
    seed=BOOTSTRAP_SEED,
):
    risk_new = np.asarray(risk_new, dtype=float).reshape(-1)
    risk_reference = np.asarray(risk_reference, dtype=float).reshape(-1)
    original_new = harrell_c(eval_frame, risk_new)
    original_reference = harrell_c(eval_frame, risk_reference)
    rng = np.random.default_rng(seed)
    n = len(eval_frame)
    values = []
    for _ in range(int(bootstrap_n)):
        idx = rng.integers(0, n, size=n)
        boot_frame = eval_frame.iloc[idx].reset_index(drop=True)
        if boot_frame["Event"].nunique() < 2:
            continue
        try:
            values.append(
                harrell_c(boot_frame, risk_new[idx])
                - harrell_c(boot_frame, risk_reference[idx])
            )
        except Exception:
            continue
    lower, upper = quantile_ci(values)
    return {
        "Harrell_C_New": original_new,
        "Harrell_C_Reference": original_reference,
        "Delta_Harrell_C": original_new - original_reference,
        "Delta_Harrell_C_Lower95": lower,
        "Delta_Harrell_C_Upper95": upper,
        "Bootstrap_N_Used": int(len(values)),
    }

def fit_absolute_risk_calibrator(train_frame, train_risk, label):
    train_risk=np.asarray(train_risk,dtype=float).reshape(-1)
    center=float(np.mean(train_risk)); scale=float(np.std(train_risk,ddof=0))
    if not np.isfinite(scale) or scale<=0: raise ValueError(f"Degenerate training risk for {label}")
    calibration_frame=pd.DataFrame({"Time":train_frame["Time"].astype(float).to_numpy(),"Event":train_frame["Event"].astype(int).to_numpy(),"Risk_Score":(train_risk-center)/scale})
    calibrator=CoxPHFitter(penalizer=0.0,l1_ratio=0.0)
    with warnings.catch_warnings():
        warnings.simplefilter("error",ConvergenceWarning)
        calibrator.fit(calibration_frame,duration_col="Time",event_col="Event",show_progress=False)
    return {"label":label,"model":calibrator,"center":center,"scale":scale,"method":"train-fitted Cox recalibration of standardized raw model score"}

def predict_absolute_risk(calibrator, raw_risk, horizon):
    score=(np.asarray(raw_risk,dtype=float).reshape(-1)-calibrator["center"])/calibrator["scale"]
    survival=calibrator["model"].predict_survival_function(pd.DataFrame({"Risk_Score":score}),times=[float(horizon)])
    return np.clip(1.0-survival.iloc[0].to_numpy(dtype=float),0.0,1.0)

def prediction_matrix(calibrator, raw_risk, times):
    return np.column_stack([
        predict_absolute_risk(calibrator, raw_risk, float(time))
        for time in times
    ])

def fit_censoring_km(frame):
    km = KaplanMeierFitter()
    km.fit(
        frame["Time"].astype(float).to_numpy(),
        event_observed=1 - frame["Event"].astype(int).to_numpy(),
        label="G_censoring",
    )
    return km

def censor_survival(km, times, left_limit=False, minimum=1e-8):
    times = np.asarray(times, dtype=float)
    if left_limit:
        times = np.nextafter(times, -np.inf)
        times = np.maximum(times, 0.0)
    values = np.asarray(km.predict(times), dtype=float).reshape(-1)
    return np.clip(values, minimum, 1.0)

def horizon_support_audit(train_frame, eval_frame, horizons=HORIZONS):
    train_censoring_km = fit_censoring_km(train_frame)
    eval_censoring_km = fit_censoring_km(eval_frame)
    rows = []
    for horizon in horizons:
        horizon = float(horizon)
        train_time = train_frame["Time"].astype(float).to_numpy()
        train_event = train_frame["Event"].astype(int).to_numpy()
        eval_time = eval_frame["Time"].astype(float).to_numpy()
        eval_event = eval_frame["Event"].astype(int).to_numpy()
        rows.append({
            "Horizon_Years": horizon,
            "Train_Event_By_Horizon_N": int(((train_event == 1) & (train_time <= horizon)).sum()),
            "Train_Observed_Beyond_Horizon_N": int((train_time > horizon).sum()),
            "Test_Event_By_Horizon_N": int(((eval_event == 1) & (eval_time <= horizon)).sum()),
            "Test_Observed_Beyond_Horizon_N": int((eval_time > horizon).sum()),
            "Train_Censoring_Survival_Gt": float(censor_survival(train_censoring_km, [horizon])[0]),
            "Test_Censoring_Survival_Gt": float(censor_survival(eval_censoring_km, [horizon])[0]),
            "Train_Max_Followup": float(np.nanmax(train_time)),
            "Test_Max_Followup": float(np.nanmax(eval_time)),
        })
    audit = pd.DataFrame(rows)
    unsupported = audit.loc[
        (audit["Train_Event_By_Horizon_N"] == 0)
        | (audit["Train_Observed_Beyond_Horizon_N"] == 0)
        | (audit["Test_Event_By_Horizon_N"] == 0)
        | (audit["Test_Observed_Beyond_Horizon_N"] == 0)
        | (audit["Train_Censoring_Survival_Gt"] <= 1e-4)
        | (audit["Test_Censoring_Survival_Gt"] <= 1e-4)
    ]
    if not unsupported.empty:
        raise RuntimeError(
            "Insufficient event/censoring support at one or more requested horizons:\n"
            + unsupported.to_string(index=False)
        )
    return audit

def ipcw_nri_idi_once(
    eval_frame,
    pred_reference,
    pred_new,
    horizon,
    cutpoints=NRI_CUTPOINTS,
    censoring_km=None,
):
    pred_reference = np.asarray(pred_reference, dtype=float).reshape(-1)
    pred_new = np.asarray(pred_new, dtype=float).reshape(-1)
    time = eval_frame["Time"].astype(float).to_numpy()
    event = eval_frame["Event"].astype(int).to_numpy()
    horizon = float(horizon)
    km = fit_censoring_km(eval_frame) if censoring_km is None else censoring_km

    cases = (event == 1) & (time <= horizon)
    controls = time > horizon
    if cases.sum() == 0 or controls.sum() == 0:
        raise RuntimeError(f"No IPCW cases or controls at horizon={horizon}.")

    case_weights = np.zeros(len(eval_frame), dtype=float)
    control_weights = np.zeros(len(eval_frame), dtype=float)
    case_weights[cases] = 1.0 / censor_survival(km, time[cases], left_limit=True)
    g_horizon = float(censor_survival(km, [horizon])[0])
    control_weights[controls] = 1.0 / g_horizon

    case_weights /= case_weights.sum()
    control_weights /= control_weights.sum()
    delta_prediction = pred_new - pred_reference

    case_up = float(np.sum(case_weights * (delta_prediction > 0)))
    case_down = float(np.sum(case_weights * (delta_prediction < 0)))
    control_up = float(np.sum(control_weights * (delta_prediction > 0)))
    control_down = float(np.sum(control_weights * (delta_prediction < 0)))
    continuous_nri = (case_up - case_down) + (control_down - control_up)

    bins = np.asarray([-np.inf] + list(cutpoints) + [np.inf], dtype=float)
    category_reference = np.digitize(pred_reference, bins, right=False)
    category_new = np.digitize(pred_new, bins, right=False)
    category_delta = category_new - category_reference
    categorical_nri = (
        float(np.sum(case_weights * (category_delta > 0)))
        - float(np.sum(case_weights * (category_delta < 0)))
        + float(np.sum(control_weights * (category_delta < 0)))
        - float(np.sum(control_weights * (category_delta > 0)))
    )

    if not NRI_CUTPOINT_SOURCE:
        categorical_nri = np.nan

    discrimination_new = (
        float(np.sum(case_weights * pred_new))
        - float(np.sum(control_weights * pred_new))
    )
    discrimination_reference = (
        float(np.sum(case_weights * pred_reference))
        - float(np.sum(control_weights * pred_reference))
    )
    idi = discrimination_new - discrimination_reference
    return {
        "NRI_Categorical": categorical_nri,
        "NRI_Continuous": continuous_nri,
        "IDI": idi,
        "Censoring_Survival_At_Horizon": g_horizon,
        "Case_N": int(cases.sum()),
        "Control_N": int(controls.sum()),
        "Censored_Before_Horizon_N": int((~cases & ~controls).sum()),
    }

def bootstrap_ipcw_nri_idi(
    eval_frame,
    pred_reference,
    pred_new,
    horizon,
    cutpoints=NRI_CUTPOINTS,
    bootstrap_n=BOOTSTRAP_N,
    seed=BOOTSTRAP_SEED,
):
    original = ipcw_nri_idi_once(
        eval_frame,
        pred_reference,
        pred_new,
        horizon,
        cutpoints=cutpoints,
    )
    pred_reference = np.asarray(pred_reference, dtype=float).reshape(-1)
    pred_new = np.asarray(pred_new, dtype=float).reshape(-1)
    rng = np.random.default_rng(seed)
    n = len(eval_frame)
    values = {"NRI_Categorical": [], "NRI_Continuous": [], "IDI": []}
    for _ in range(int(bootstrap_n)):
        idx = rng.integers(0, n, size=n)
        boot_frame = eval_frame.iloc[idx].reset_index(drop=True)
        try:
            estimate = ipcw_nri_idi_once(
                boot_frame,
                pred_reference[idx],
                pred_new[idx],
                horizon,
                cutpoints=cutpoints,
            )
            for metric in values:
                values[metric].append(estimate[metric])
        except Exception:
            continue

    output = dict(original)
    for metric, metric_values in values.items():
        lower, upper = quantile_ci(metric_values)
        output[f"{metric}_Lower95"] = lower
        output[f"{metric}_Upper95"] = upper
    output["Bootstrap_N_Used"] = int(min(len(v) for v in values.values()))
    return output

def subject_level_ipcw_ibs(
    train_frame,
    eval_frame,
    calibrator,
    eval_risk,
    horizon,
    grid_n=IBS_GRID_N,
):
    horizon = float(horizon)
    times = np.linspace(0.0, horizon, int(grid_n))
    predicted_risk = prediction_matrix(calibrator, eval_risk, times)
    time = eval_frame["Time"].astype(float).to_numpy()
    event = eval_frame["Event"].astype(int).to_numpy()
    censor_km = fit_censoring_km(train_frame)
    contributions = np.zeros_like(predicted_risk, dtype=float)

    for j, point in enumerate(times):
        y = ((event == 1) & (time <= point)).astype(float)
        weights = np.zeros(len(eval_frame), dtype=float)
        cases = (event == 1) & (time <= point)
        controls = time > point
        if cases.any():
            weights[cases] = 1.0 / censor_survival(
                censor_km,
                time[cases],
                left_limit=True,
            )
        if controls.any():
            weights[controls] = 1.0 / float(censor_survival(censor_km, [point])[0])
        contributions[:, j] = weights * (y - predicted_risk[:, j]) ** 2

    if hasattr(np, "trapezoid"):
        subject_ibs = np.trapezoid(contributions, times, axis=1) / horizon
    else:
        subject_ibs = np.trapz(contributions, times, axis=1) / horizon
    return {
        "IBS": float(np.mean(subject_ibs)),
        "Subject_IBS": subject_ibs,
        "Times": times,
    }

def paired_bootstrap_delta_ibs(
    new_ibs,
    reference_ibs,
    bootstrap_n=BOOTSTRAP_N,
    seed=BOOTSTRAP_SEED,
):
    paired = (
        np.asarray(new_ibs["Subject_IBS"], dtype=float)
        - np.asarray(reference_ibs["Subject_IBS"], dtype=float)
    )
    rng = np.random.default_rng(seed)
    n = len(paired)
    values = []
    for _ in range(int(bootstrap_n)):
        idx = rng.integers(0, n, size=n)
        values.append(float(np.mean(paired[idx])))
    lower, upper = quantile_ci(values)
    return {
        "IBS_New": float(new_ibs["IBS"]),
        "IBS_Reference": float(reference_ibs["IBS"]),
        "Delta_IBS": float(np.mean(paired)),
        "Delta_IBS_Lower95": lower,
        "Delta_IBS_Upper95": upper,
        "Bootstrap_N_Used": int(len(values)),
    }

def km_observed_risk(frame, horizon):
    if len(frame) == 0:
        return np.nan
    km = KaplanMeierFitter()
    km.fit(
        frame["Time"].astype(float).to_numpy(),
        event_observed=frame["Event"].astype(int).to_numpy(),
    )
    return float(1.0 - km.predict(float(horizon)))

def calibration_by_quantile(eval_frame, predicted_risk, horizon, groups=10):
    temporary = eval_frame[["Time", "Event"]].reset_index(drop=True).copy()
    temporary["Predicted_Risk"] = np.asarray(predicted_risk, dtype=float).reshape(-1)
    temporary["Risk_Group"] = pd.qcut(
        temporary["Predicted_Risk"].rank(method="first"),
        q=int(groups),
        labels=False,
        duplicates="drop",
    )
    rows = []
    for group, data in temporary.groupby("Risk_Group"):
        rows.append({
            "Risk_Group": int(group) + 1,
            "N": int(len(data)),
            "Predicted_Risk": float(data["Predicted_Risk"].mean()),
            "Observed_KM_Risk": km_observed_risk(data, horizon),
        })
    return pd.DataFrame(rows)

def _safe_dca_column(name):
    text = re.sub(r"[^A-Za-z0-9_]+", "_", str(name)).strip("_")
    return "model_" + text.lower()

def survival_dca(eval_frame, prediction_dict, horizon, thresholds=DCA_THRESHOLDS):
    data = eval_frame[["Time", "Event"]].reset_index(drop=True).copy()
    model_columns = []
    label_map = {}
    for label, prediction in prediction_dict.items():
        column = _safe_dca_column(label)
        data[column] = np.asarray(prediction, dtype=float).reshape(-1)
        model_columns.append(column)
        label_map[column] = label

    result = dcurves_dca(
        data=data,
        outcome="Event",
        modelnames=model_columns,
        thresholds=np.asarray(thresholds, dtype=float),
        time_to_outcome_col="Time",
        time=float(horizon),
    ).copy()
    result.columns = [str(column).lower() for column in result.columns]
    rename = {}
    for candidate in ["model", "models", "label"]:
        if candidate in result.columns:
            rename[candidate] = "Strategy"
            break
    for candidate in ["threshold", "thresholds"]:
        if candidate in result.columns:
            rename[candidate] = "Threshold"
            break
    for candidate in ["net_benefit", "net benefit", "netbenefit"]:
        if candidate in result.columns:
            rename[candidate] = "Net_Benefit"
            break
    result = result.rename(columns=rename)
    required = {"Strategy", "Threshold", "Net_Benefit"}
    if not required.issubset(result.columns):
        raise RuntimeError(
            "Unexpected dcurves output columns: " + ", ".join(result.columns)
        )
    result["Strategy"] = result["Strategy"].replace(label_map).replace({
        "all": "Treat all",
        "none": "Treat none",
        "All": "Treat all",
        "None": "Treat none",
    })
    result["Horizon_Years"] = float(horizon)
    result["Split"] = "test"
    result["DCA_Method"] = "dcurves survival DCA; no external IPCW multiplication"
    return result

def draw_forest(
    frame,
    estimate,
    lower,
    upper,
    label,
    title,
    xlabel,
    output_dir,
    stem,
    color="#3C5488",
    null=0.0,
    figsize=(8.2, 4.6),
):
    plot = frame.reset_index(drop=True).copy()
    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(len(plot))
    ax.axvline(float(null), color=CNS["gray"], lw=1.0, zorder=0)
    for i, row in plot.iterrows():
        value = float(row[estimate])
        low = float(row[lower])
        high = float(row[upper])
        row_color = row[color] if isinstance(color, str) and color in plot.columns else color
        ax.errorbar(
            value,
            i,
            xerr=[[max(0.0, value - low)], [max(0.0, high - value)]],
            fmt="o",
            color=row_color,
            ecolor=row_color,
            elinewidth=1.5,
            capsize=3,
            markersize=6.5,
        )
    ax.set_yticks(y)
    ax.set_yticklabels(plot[label].astype(str))
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.grid(axis="x", color=CNS["light_gray"], lw=0.8)
    clean_axes(ax)
    fig.tight_layout()
    save_figure(fig, output_dir, stem)
    plt.show()

def draw_dca(dca_frame, title, output_dir, stem, colors, figsize=(8.2, 5.0)):
    fig, ax = plt.subplots(figsize=figsize)
    preferred_order = list(colors)
    available = list(dict.fromkeys(dca_frame["Strategy"].astype(str)))
    order = [item for item in preferred_order if item in available]
    order += [item for item in available if item not in order]
    for strategy in order:
        data = dca_frame.loc[dca_frame["Strategy"].astype(str) == strategy].sort_values("Threshold")
        if data.empty:
            continue
        style = "--" if strategy == "Treat all" else ":" if strategy == "Treat none" else "-"
        width = 1.5 if strategy in {"Treat all", "Treat none"} else 2.1
        ax.plot(
            data["Threshold"],
            data["Net_Benefit"],
            color=colors.get(strategy, CNS["gray"]),
            linestyle=style,
            linewidth=width,
            label=strategy,
        )
    ax.set_xlabel("Predicted-risk threshold")
    ax.set_ylabel("Net benefit")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.grid(color=CNS["light_gray"], lw=0.8)
    ax.legend(frameon=False, fontsize=11.0, ncol=2)
    clean_axes(ax)
    fig.tight_layout()
    save_figure(fig, output_dir, stem)
    plt.show()


In [ ]:
from pathlib import Path
import os
import sklearn
import sksurv
from sksurv.linear_model import CoxnetSurvivalAnalysis

SHARED_NOTEBOOK = os.environ.get(
    "CHD_SHARED_NOTEBOOK",
    str(Path.cwd() / "00_shared_config.ipynb"),
)
if not Path(SHARED_NOTEBOOK).exists():
    raise FileNotFoundError(
        f"Shared notebook not found: {SHARED_NOTEBOOK}. "
        "Place this notebook beside 00_shared_config.ipynb "
        "or set CHD_SHARED_NOTEBOOK."
    )
get_ipython().run_line_magic("run", f'"{SHARED_NOTEBOOK}"')

(
    df_train,
    df_val,
    df_test,
    final_feature_sets,
    pipeline_manifest,
) = validate_pipeline_artifacts(load_frames=True)

RESULT3_ROOT = ensure_dir(os.path.join(
    SENSITIVITY_OUTPUT_DIR,
    "04_results",
    "result3_cox_vs_elasticnet_ipcw",
))
RESULT3_FIGURE_DIR = ensure_dir(os.path.join(RESULT3_ROOT, "figures"))
RESULT3_SOURCE_DIR = ensure_dir(os.path.join(RESULT3_ROOT, "source_data"))
RESULT3_CACHE_DIR = ensure_dir(os.path.join(RESULT3_ROOT, "elasticnet_cache"))
print("Result 3 output:", RESULT3_ROOT)


In [ ]:
# Optimized prediction-only ordinary Cox PH fitting for Result 3.
#
# Statistical estimand is unchanged: penalizer=0 and Efron ties. Robust
# sandwich SEs are intentionally omitted because Result 3 compares prediction
# risks and does not make coefficient-level inference. Existing signed v2
# prediction caches remain reusable because robust=True changes the covariance
# estimate, not the fitted partial-likelihood point estimate.
import time
from lifelines.exceptions import ConvergenceWarning as LifelinesConvergenceWarning
from sklearn.exceptions import ConvergenceWarning as SklearnConvergenceWarning
from sksurv.linear_model import CoxPHSurvivalAnalysis

RESULT3_COX_SHOW_PROGRESS = os.environ.get("CHD_RESULT3_COX_SHOW_PROGRESS", "1").strip().lower() not in {"0", "false", "no"}
RESULT3_COX_BATCH_MODE = os.environ.get("CHD_RESULT3_COX_BATCH_MODE", "1").strip().lower() not in {"0", "false", "no"}
RESULT3_COX_STEP_SIZE = float(os.environ.get("CHD_RESULT3_COX_STEP_SIZE", "0.5"))
RESULT3_COX_MAX_STEPS = int(os.environ.get("CHD_RESULT3_COX_MAX_STEPS", "120"))
RESULT3_COX_PRECISION = float(os.environ.get("CHD_RESULT3_COX_PRECISION", "1e-7"))
RESULT3_COX_SKSURV_N_ITER = int(os.environ.get("CHD_RESULT3_COX_SKSURV_N_ITER", "200"))
RESULT3_COX_SKSURV_TOL = float(os.environ.get("CHD_RESULT3_COX_SKSURV_TOL", "1e-9"))
RESULT3_COX_CODE_TAG = "result3_unpenalized_cox_prediction_v3"

def _result3_read_coefficient_map(feature_set_name):
    path = os.path.join(
        model_result_dir("Cox PH"),
        f"{safe_filename(feature_set_name)}_coefficients.csv",
    )
    if not os.path.exists(path):
        return {}
    table = pd.read_csv(path)
    variable_column = "Variable" if "Variable" in table.columns else None
    coefficient_column = "coef" if "coef" in table.columns else (
        "Coefficient" if "Coefficient" in table.columns else None
    )
    if variable_column is None or coefficient_column is None:
        return {}
    usable = table[[variable_column, coefficient_column]].dropna()
    return dict(zip(usable[variable_column].astype(str), usable[coefficient_column].astype(float)))

def _result3_engine_hyperparameters(engine):
    common = {
        "penalizer": 0.0,
        "robust_se": False,
        "ties": "efron",
        "prediction_only_fit": True,
        "engine": engine,
    }
    if engine == "lifelines":
        common.update({
            "l1_ratio": 0.0,
            "batch_mode": bool(RESULT3_COX_BATCH_MODE),
            "step_size": RESULT3_COX_STEP_SIZE,
            "max_steps": RESULT3_COX_MAX_STEPS,
            "precision": RESULT3_COX_PRECISION,
        })
    elif engine == "sksurv":
        common.update({
            "alpha": 0.0,
            "n_iter": RESULT3_COX_SKSURV_N_ITER,
            "tol": RESULT3_COX_SKSURV_TOL,
        })
    else:
        raise KeyError(engine)
    return common

def _result3_load_cached_row(feature_set_name, actual_columns):
    # First prefer a v3 prediction-only cache from either exact Cox engine.
    for engine in ["lifelines", "sksurv"]:
        hyperparameters = _result3_engine_hyperparameters(engine)
        if risk_cache_matches(
            "Cox PH", feature_set_name, actual_columns,
            hyperparameters, RESULT3_COX_CODE_TAG,
        ):
            row = pd.read_csv(model_result_path("Cox PH", feature_set_name)).iloc[0].to_dict()
            return row, engine, "v3_signed_cache"

    # Migration path for the already completed 22/23/68-variable fits. Robust
    # covariance estimation does not alter their saved risk vectors.
    legacy_hyperparameters = {
        "penalizer": 0.0, "l1_ratio": 0.0,
        "robust_se": True, "ties": "efron",
    }
    if risk_cache_matches(
        "Cox PH", feature_set_name, actual_columns,
        legacy_hyperparameters, "unpenalized_cox_v2",
    ):
        row = pd.read_csv(model_result_path("Cox PH", feature_set_name)).iloc[0].to_dict()
        return row, "lifelines", "legacy_v2_prediction_cache"
    return None, None, None

def _result3_fit_lifelines(fit_frame, X_train, X_val, X_test, initial_point):
    model = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
    options = {
        "step_size": RESULT3_COX_STEP_SIZE,
        "max_steps": RESULT3_COX_MAX_STEPS,
        "precision": RESULT3_COX_PRECISION,
    }
    with warnings.catch_warnings():
        warnings.simplefilter("error", LifelinesConvergenceWarning)
        model.fit(
            fit_frame,
            duration_col="Time",
            event_col="Event",
            robust=False,
            batch_mode=RESULT3_COX_BATCH_MODE,
            show_progress=RESULT3_COX_SHOW_PROGRESS,
            initial_point=initial_point,
            fit_options=options,
        )
    risks = {
        "train_risk": model.predict_log_partial_hazard(X_train).to_numpy().reshape(-1),
        "val_risk": model.predict_log_partial_hazard(X_val).to_numpy().reshape(-1),
        "test_risk": model.predict_log_partial_hazard(X_test).to_numpy().reshape(-1),
    }
    summary = model.summary.reset_index()
    summary = summary.rename(columns={summary.columns[0]: "Variable"})
    baseline = model.baseline_cumulative_hazard_.reset_index()
    return model, risks, summary, baseline

def _result3_fit_sksurv(X_train_np, X_val_np, X_test_np, actual_columns):
    # Exact unpenalized Cox fallback: alpha=0, same Efron tie handling.
    model = CoxPHSurvivalAnalysis(
        alpha=0.0,
        ties="efron",
        n_iter=RESULT3_COX_SKSURV_N_ITER,
        tol=RESULT3_COX_SKSURV_TOL,
        verbose=1 if RESULT3_COX_SHOW_PROGRESS else 0,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("error", SklearnConvergenceWarning)
        model.fit(X_train_np, make_surv(df_train))
    risks = {
        "train_risk": np.asarray(model.predict(X_train_np), dtype=float),
        "val_risk": np.asarray(model.predict(X_val_np), dtype=float),
        "test_risk": np.asarray(model.predict(X_test_np), dtype=float),
    }
    summary = pd.DataFrame({
        "Variable": actual_columns,
        "coef": np.asarray(model.coef_, dtype=float),
        "SE_Available": False,
        "Inference_Use": "No; prediction-only Result 3 fit",
    })
    baseline = pd.DataFrame({
        "timeline": np.asarray(model.unique_times_, dtype=float),
        "baseline cumulative hazard": np.asarray(
            model.cum_baseline_hazard_(model.unique_times_), dtype=float
        ),
    })
    return model, risks, summary, baseline

def fit_result3_unpenalized_cox_feature_sets(
    train_frame,
    val_frame,
    test_frame,
    feature_sets,
    feature_names,
    warm_start_feature_set=None,
    runtime_audit_filename="result3_primary_cox_runtime_audit.csv",
):
    global df_train, df_val, df_test, final_feature_sets
    df_train, df_val, df_test = train_frame, val_frame, test_frame
    final_feature_sets = feature_sets
    feature_names = list(feature_names)

    # Column-wise imputation/scaling is invariant to the other columns, so one
    # union preprocessing pass is exactly equivalent to four repeated passes.
    union_columns = list(dict.fromkeys(
        column for feature_set_name in feature_names
        for column in feature_sets[feature_set_name]
    ))
    X_union_train, X_union_val, X_union_test, _ = prepare_X_train_val_test(
        train_frame, val_frame, test_frame, union_columns, scale=True
    )
    outcome_frame = train_frame[["Time", "Event"]].reset_index(drop=True).copy()
    rows = []
    audit_rows = []
    warm_coefficients = (
        _result3_read_coefficient_map(warm_start_feature_set)
        if warm_start_feature_set else {}
    )

    for feature_set_name in feature_names:
        actual_columns = list(feature_sets[feature_set_name])
        X_train = X_union_train.loc[:, actual_columns].copy()
        X_val = X_union_val.loc[:, actual_columns].copy()
        X_test = X_union_test.loc[:, actual_columns].copy()
        X_train_np = np.ascontiguousarray(X_train.to_numpy(dtype=np.float64, copy=False))
        X_val_np = np.ascontiguousarray(X_val.to_numpy(dtype=np.float64, copy=False))
        X_test_np = np.ascontiguousarray(X_test.to_numpy(dtype=np.float64, copy=False))

        cached_row, cached_engine, cache_status = _result3_load_cached_row(
            feature_set_name, actual_columns
        )
        if cached_row is not None:
            rows.append(cached_row)
            warm_coefficients = _result3_read_coefficient_map(feature_set_name)
            log_progress(
                f"RESUME Cox PH | {feature_set_name} | {cache_status} | engine={cached_engine}"
            )
            audit_rows.append({
                "Feature_Set": feature_set_name,
                "Feature_N": len(actual_columns),
                "Fit_Engine": cached_engine,
                "Cache_Status": cache_status,
                "Robust_SE_Computed": bool(cached_row.get("Robust_SE", False)),
                "Convergence_Status": "signed_cache_valid",
                "Elapsed_Minutes": 0.0,
                "Lifelines_Error": "",
            })
            continue

        fit_frame = pd.concat([outcome_frame, X_train.reset_index(drop=True)], axis=1).copy()
        initial_point = None
        if warm_coefficients:
            candidate = np.asarray(
                [warm_coefficients.get(column, 0.0) for column in actual_columns],
                dtype=float,
            )
            if np.isfinite(candidate).all() and np.any(candidate != 0):
                initial_point = candidate

        started = time.perf_counter()
        lifelines_error = ""
        engine = "lifelines"
        log_progress(
            f"START optimized Cox PH | {feature_set_name} | features={len(actual_columns)} "
            f"| robust_se=False | batch_mode={RESULT3_COX_BATCH_MODE} "
            f"| warm_start={initial_point is not None}"
        )
        try:
            model, risks, summary, baseline = _result3_fit_lifelines(
                fit_frame, X_train, X_val, X_test, initial_point
            )
        except Exception as error:
            lifelines_error = repr(error)
            engine = "sksurv"
            log_progress(
                f"RETRY exact Cox PH with sksurv | {feature_set_name} | "
                f"lifelines failed: {lifelines_error}"
            )
            try:
                model, risks, summary, baseline = _result3_fit_sksurv(
                    X_train_np, X_val_np, X_test_np, actual_columns
                )
            except Exception as fallback_error:
                raise RuntimeError(
                    f"Both exact unpenalized Cox engines failed for {feature_set_name}. "
                    f"lifelines={lifelines_error}; sksurv={fallback_error!r}. "
                    "Do not silently relabel a ridge fit as ordinary Cox; report the "
                    "non-estimability and retain Elastic-net Cox as the required sensitivity."
                ) from fallback_error

        for split_name, vector in risks.items():
            vector = np.asarray(vector, dtype=float).reshape(-1)
            if not np.isfinite(vector).all() or np.std(vector) <= 0:
                raise RuntimeError(
                    f"Invalid {split_name} from {engine} Cox fit: {feature_set_name}"
                )
            risks[split_name] = vector

        hyperparameters = _result3_engine_hyperparameters(engine)
        signature = save_risk_record(
            "Cox PH", feature_set_name, risks, actual_columns,
            hyperparameters, RESULT3_COX_CODE_TAG,
        )
        row = make_result_row(
            MAIN_SCENARIO_NAME, "Cox PH", feature_set_name, actual_columns,
            risks["train_risk"], risks["val_risk"], risks["test_risk"],
        )
        elapsed_minutes = (time.perf_counter() - started) / 60.0
        row.update({
            "COX_PENALIZER": 0.0,
            "Robust_SE": False,
            "Prediction_Only_Fit": True,
            "Fit_Engine": engine,
            "Tie_Method": "efron",
            "Convergence_Status": "converged",
            "Model_Signature": signature,
            "Elapsed_Minutes": elapsed_minutes,
        })
        atomic_write_csv(
            pd.DataFrame([row]),
            model_result_path("Cox PH", feature_set_name),
            index=False,
        )
        summary.insert(0, "Feature_Set", feature_set_name)
        summary["Fit_Engine"] = engine
        summary["Prediction_Only_Fit"] = True
        atomic_write_csv(
            summary,
            os.path.join(
                model_result_dir("Cox PH"),
                f"{safe_filename(feature_set_name)}_coefficients.csv",
            ),
            index=False,
        )
        atomic_write_csv(
            baseline,
            os.path.join(
                model_result_dir("Cox PH"),
                f"{safe_filename(feature_set_name)}_baseline_cumulative_hazard.csv",
            ),
            index=False,
        )
        rows.append(row)
        warm_coefficients = dict(
            zip(actual_columns, np.asarray(model.params_ if engine == "lifelines" else model.coef_, dtype=float))
        )
        audit_rows.append({
            "Feature_Set": feature_set_name,
            "Feature_N": len(actual_columns),
            "Fit_Engine": engine,
            "Cache_Status": "fitted_v3",
            "Robust_SE_Computed": False,
            "Convergence_Status": "converged",
            "Elapsed_Minutes": elapsed_minutes,
            "Lifelines_Error": lifelines_error,
        })
        log_progress(
            f"DONE optimized Cox PH | {feature_set_name} | engine={engine} "
            f"| elapsed={elapsed_minutes:.2f} min | test Harrell C={row['Test_C_index']:.6f}"
        )

    audit = pd.DataFrame(audit_rows)
    atomic_write_csv(
        audit,
        os.path.join(RESULT3_SOURCE_DIR, runtime_audit_filename),
        index=False,
    )
    return pd.DataFrame(rows)

RESULT3_PRIMARY_COX_FEATURES = [
    "Clinical_plus_PRS",
    "Clinical_plus_PRS_plus_NMR_score",
    "Clinical_plus_PRS_plus_SpearmanNMR",
    "Clinical_plus_PRS_plus_Raw106NMR",
]
result3_cox_fit = fit_result3_unpenalized_cox_feature_sets(
    df_train,
    df_val,
    df_test,
    final_feature_sets,
    RESULT3_PRIMARY_COX_FEATURES,
)
atomic_write_csv(
    result3_cox_fit,
    os.path.join(TABLE_DIR, "result3_primary_unpenalized_cox.csv"),
    index=False,
)


In [ ]:
from IPython.display import display as ipy_display
RESULT3_BASE_FEATURE = "Clinical_plus_PRS"
RESULT3_REPRESENTATIONS = [
    ("NMR_score", "Clinical_plus_PRS_plus_NMR_score"),
    ("Spearman-pruned NMR", "Clinical_plus_PRS_plus_SpearmanNMR"),
    ("Raw106 NMR", "Clinical_plus_PRS_plus_Raw106NMR"),
]
RESULT3_FEATURES = [RESULT3_BASE_FEATURE] + [feature for _, feature in RESULT3_REPRESENTATIONS]
RESULT3_EXPANDED_FEATURE = "Clinical_plus_PRS_plus_Full249NMR"
RESULT3_EXPANDED_LABEL = "Full249 expanded panel"
RESULT3_REQUIRED_FEATURES = RESULT3_FEATURES + [RESULT3_EXPANDED_FEATURE]
missing_feature_sets = [name for name in RESULT3_REQUIRED_FEATURES if name not in final_feature_sets]
if missing_feature_sets:
    raise KeyError("Missing Stage 01 feature sets: " + ", ".join(missing_feature_sets))

# Primary fits use the optimized exact Cox pathway defined above. Full249 ordinary
# Cox remains an optional expanded-panel diagnostic: failure is audited and cannot
# block the required Full249 Elastic-net sensitivity.
EXPANDED_COX_STATUS = "not_attempted"
EXPANDED_COX_ERROR = ""
try:
    expanded_cox_fit = fit_result3_unpenalized_cox_feature_sets(
        df_train,
        df_val,
        df_test,
        final_feature_sets,
        [RESULT3_EXPANDED_FEATURE],
        warm_start_feature_set="Clinical_plus_PRS_plus_Raw106NMR",
        runtime_audit_filename="supplement_result3_full249_cox_runtime_audit.csv",
    )
    EXPANDED_COX_STATUS = "available"
except Exception as error:
    EXPANDED_COX_STATUS = "failed_nonblocking"
    EXPANDED_COX_ERROR = repr(error)
    expanded_cox_fit = pd.DataFrame([{
        "Feature_Set": RESULT3_EXPANDED_FEATURE,
        "Analysis_Role": "expanded_panel_sensitivity",
        "Cox_Status": EXPANDED_COX_STATUS,
        "Cox_Error": EXPANDED_COX_ERROR,
    }])
    warnings.warn(
        "Full249 ordinary Cox did not converge or could not be loaded; "
        "continuing the prespecified Full249 Elastic-net sensitivity. "
        + EXPANDED_COX_ERROR
    )

EXPANDED_COX_AVAILABLE = EXPANDED_COX_STATUS == "available"
RESULT3_EXPANDED_ALGORITHMS = (
    (["Cox PH"] if EXPANDED_COX_AVAILABLE else []) + ["Elastic-net Cox"]
)
atomic_write_csv(
    expanded_cox_fit,
    os.path.join(TABLE_DIR, "supplement_result3_full249_expanded_panel_unpenalized_cox.csv"),
    index=False,
)
expanded_cox_status_audit = pd.DataFrame([{
    "Feature_Set": RESULT3_EXPANDED_FEATURE,
    "Analysis_Role": "expanded_panel_sensitivity",
    "Cox_Status": EXPANDED_COX_STATUS,
    "Cox_Available_For_Downstream_Metrics": EXPANDED_COX_AVAILABLE,
    "Cox_Error": EXPANDED_COX_ERROR,
    "Failure_Blocks_Primary_Result3": False,
    "Elasticnet_Remains_Required": True,
}])
atomic_write_csv(
    expanded_cox_status_audit,
    os.path.join(RESULT3_SOURCE_DIR, "supplement_result3_full249_expanded_panel_cox_status.csv"),
    index=False,
)

RESULT3_AVAILABLE_COX_FEATURES = RESULT3_FEATURES + (
    [RESULT3_EXPANDED_FEATURE] if EXPANDED_COX_AVAILABLE else []
)
selected_feature_sets = {
    name: final_feature_sets[name] for name in RESULT3_AVAILABLE_COX_FEATURES
}
cox_risk_store = load_all_risk_records(selected_feature_sets, ["Cox PH"])

def result3_cox_record(feature_set):
    key = (MAIN_SCENARIO_NAME, feature_set, "Cox PH")
    if key not in cox_risk_store:
        raise KeyError(f"Missing saved Cox risk record: {key}")
    return cox_risk_store[key]

for feature_set in RESULT3_AVAILABLE_COX_FEATURES:
    record = result3_cox_record(feature_set)
    if len(record["train_risk"]) != len(df_train) or len(record["test_risk"]) != len(df_test):
        raise RuntimeError(f"Risk/data length mismatch for {feature_set}")

result3_horizon_audit = horizon_support_audit(df_train, df_test, HORIZONS)
result3_horizon_audit["Evaluation_Population"] = "locked test set"
result3_horizon_audit.to_csv(
    os.path.join(RESULT3_SOURCE_DIR, "horizon_support_audit.csv"),
    index=False,
)
ipy_display(result3_horizon_audit)

competing_event_column = os.environ.get("CHD_COMPETING_EVENT_COL", "").strip()
pd.DataFrame([{
    "Competing_Event_Column_Configured": competing_event_column,
    "Competing_Event_Column_Available": bool(
        competing_event_column and competing_event_column in df_test.columns
    ),
    "Primary_Estimand": "Cause-specific time to first CHD under right censoring",
    "Warning": "A separate non-CHD death indicator is required for competing-risk evaluation.",
}]).to_csv(os.path.join(RESULT3_SOURCE_DIR, "competing_risk_data_audit.csv"), index=False)


In [ ]:
from IPython.display import display as ipy_display
import time
# Elastic-net configuration. Each l1_ratio fits a complete alpha path. Selection
# is validation-only. The final model is refit on training only so that its
# development sample matches the saved ordinary Cox PH model.
# The l1_ratio grid covers near-ridge through lasso penalties; boundary selections are audited.
EN_L1_RATIOS = [
    float(value)
    for value in os.environ.get("CHD_EN_L1_RATIOS", "1e-6,1e-5,1e-4,1e-3,0.005,0.01,0.025,0.05,0.1,0.25,0.5,0.75,0.9,0.99,1.0").split(",")
]
EN_N_ALPHAS = int(os.environ.get("CHD_EN_N_ALPHAS", "60"))
EN_ALPHA_MIN_RATIO = float(os.environ.get("CHD_EN_ALPHA_MIN_RATIO", "1e-6"))
EN_TUNING_TAU = float(os.environ.get("CHD_EN_TUNING_TAU", "10"))
EN_MAX_ITER = int(os.environ.get("CHD_EN_MAX_ITER", "100000"))
EN_TIE_TOL = float(os.environ.get("CHD_EN_TIE_TOL", "1e-6"))
EN_ALPHA_EXTENSION_FACTOR = float(os.environ.get("CHD_EN_ALPHA_EXTENSION_FACTOR", "1e-2"))
EN_ALPHA_EXTENSION_POINTS = int(os.environ.get("CHD_EN_ALPHA_EXTENSION_POINTS", "30"))
EN_ALPHA_EXTENSION_MAX_ROUNDS = int(os.environ.get("CHD_EN_ALPHA_EXTENSION_MAX_ROUNDS", "3"))
EN_ALPHA_ABSOLUTE_FLOOR = float(os.environ.get("CHD_EN_ALPHA_ABSOLUTE_FLOOR", "1e-14"))
if not 0 < EN_ALPHA_EXTENSION_FACTOR < 1:
    raise ValueError("CHD_EN_ALPHA_EXTENSION_FACTOR must be in (0, 1).")
if EN_ALPHA_EXTENSION_POINTS < 5 or EN_ALPHA_EXTENSION_MAX_ROUNDS < 1:
    raise ValueError("Adaptive alpha extension requires at least 5 points and 1 round.")
try:
    _RESULT3_VISIBLE_CPUS_EARLY = len(os.sched_getaffinity(0))
except Exception:
    _RESULT3_VISIBLE_CPUS_EARLY = os.cpu_count() or 1
RESULT3_N_JOBS = int(os.environ.get("CHD_RESULT3_N_JOBS", str(min(8, _RESULT3_VISIBLE_CPUS_EARLY))))
if RESULT3_N_JOBS < 1:
    raise ValueError("CHD_RESULT3_N_JOBS must be at least 1")
EN_BOUNDARY_POLICY = os.environ.get("CHD_EN_BOUNDARY_POLICY", "fail").strip().lower()
if EN_BOUNDARY_POLICY not in {"fail", "warn"}:
    raise ValueError("CHD_EN_BOUNDARY_POLICY must be 'fail' or 'warn'.")
if sorted(set(EN_L1_RATIOS)) != EN_L1_RATIOS or any(value <= 0 or value > 1 for value in EN_L1_RATIOS):
    raise ValueError("CHD_EN_L1_RATIOS must be unique, increasing, and within (0, 1].")
if not np.isclose(EN_L1_RATIOS[-1], 1.0):
    raise ValueError("CHD_EN_L1_RATIOS must include 1.0 as the valid pure-L1 endpoint.")

X_base_train, X_base_val, X_base_test, base_actual_columns = prepare_X_train_val_test(
    df_train,
    df_val,
    df_test,
    final_feature_sets[RESULT3_BASE_FEATURE],
    scale=True,
)
base_actual_set = set(base_actual_columns)
y_train = make_surv(df_train)

def elasticnet_signature(feature_set, actual_columns, penalty_factor):
    payload = {
        "feature_set": feature_set,
        "actual_columns": list(actual_columns),
        "penalty_factor": np.asarray(penalty_factor, dtype=float).tolist(),
        "l1_ratios": EN_L1_RATIOS,
        "n_alphas": EN_N_ALPHAS,
        "alpha_min_ratio": EN_ALPHA_MIN_RATIO,
        "tuning_metric": f"validation Uno C tau={EN_TUNING_TAU}",
        "final_fit": "training only",
        "l1_ratio_search_boundary_check": True,
        "boundary_policy": EN_BOUNDARY_POLICY,
        "tie_tolerance": EN_TIE_TOL,
        "train_eid_sha256": eid_digest(df_train),
        "val_eid_sha256": eid_digest(df_val),
        "test_eid_sha256": eid_digest(df_test),
        "sksurv_version": sksurv.__version__,
        "sklearn_version": sklearn.__version__,
    }
    encoded = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest(), payload

def enforce_elasticnet_boundary_policy(feature_set, metadata):
    # Alpha endpoints and the near-zero l1_ratio endpoint indicate an expandable
    # search boundary. l1_ratio=1.0 is different: it is the closed pure-L1
    # parameter endpoint and cannot be extended further.
    blocking_boundary_fields = [
        "selected_at_alpha_min_boundary", "selected_at_alpha_max_boundary",
        "selected_at_l1_ratio_min_boundary",
    ]
    accepted_parameter_endpoint = bool(
        metadata.get("selected_at_l1_ratio_max_boundary", False)
        and np.isclose(float(metadata.get("selected_l1_ratio", np.nan)), 1.0)
    )
    metadata["accepted_parameter_endpoint"] = accepted_parameter_endpoint
    if accepted_parameter_endpoint:
        print(f"{feature_set}: selected l1_ratio=1.0; accepted pure-L1 parameter endpoint.", flush=True)
    hit = [field for field in blocking_boundary_fields if bool(metadata.get(field, False))]
    if not hit:
        return
    message = (
        f"{feature_set}: selected Elastic-net hyperparameter remains on an expandable search boundary: {hit}. "
        "Expand the grid before reporting this sensitivity analysis."
    )
    if EN_BOUNDARY_POLICY == "fail":
        raise RuntimeError(message)
    warnings.warn(message)

def elasticnet_cache_paths(feature_set):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", feature_set)
    return {
        "risk": os.path.join(RESULT3_CACHE_DIR, safe + "_risks.npz"),
        "metadata": os.path.join(RESULT3_CACHE_DIR, safe + "_metadata.json"),
        "tuning": os.path.join(RESULT3_CACHE_DIR, safe + "_validation_tuning.csv"),
        "coefficients": os.path.join(RESULT3_CACHE_DIR, safe + "_coefficients.csv"),
    }

def _fit_and_score_elasticnet_path(
    feature_set,
    X_train_np,
    X_val_np,
    penalty_factor,
    l1_ratio,
    search_stage,
    extension_round=0,
    explicit_alphas=None,
):
    path_started = time.perf_counter()
    if explicit_alphas is None:
        requested_alpha_n = EN_N_ALPHAS
        model = CoxnetSurvivalAnalysis(
            n_alphas=EN_N_ALPHAS,
            alpha_min_ratio=EN_ALPHA_MIN_RATIO,
            l1_ratio=float(l1_ratio),
            penalty_factor=penalty_factor,
            normalize=False,
            max_iter=EN_MAX_ITER,
            tol=1e-7,
            fit_baseline_model=False,
        )
    else:
        requested = np.asarray(explicit_alphas, dtype=float)
        requested = np.unique(requested[np.isfinite(requested) & (requested > 0)])[::-1]
        if len(requested) < 2:
            raise RuntimeError(
                f"Adaptive alpha extension produced fewer than two values: {feature_set}"
            )
        requested_alpha_n = len(requested)
        model = CoxnetSurvivalAnalysis(
            alphas=requested,
            l1_ratio=float(l1_ratio),
            penalty_factor=penalty_factor,
            normalize=False,
            max_iter=EN_MAX_ITER,
            tol=1e-7,
            fit_baseline_model=False,
        )

    print(
        f"[{feature_set}] START {search_stage} | l1_ratio={l1_ratio:g} "
        f"| requested_alphas={requested_alpha_n}",
        flush=True,
    )
    model.fit(X_train_np, y_train)
    alpha_values = np.asarray(model.alphas_, dtype=float)
    rows = []
    for alpha_index, alpha in enumerate(alpha_values):
        try:
            validation_risk = model.predict(X_val_np, alpha=float(alpha))
            validation_uno = uno_c(
                df_train,
                df_val,
                validation_risk,
                EN_TUNING_TAU,
            )
            status = "ok"
        except Exception as error:
            validation_uno = np.nan
            status = f"failed: {error}"
        rows.append({
            "Feature_Set": feature_set,
            "L1_Ratio": float(l1_ratio),
            "Alpha": float(alpha),
            "Alpha_Index": int(alpha_index),
            "Alpha_Path_N": int(len(alpha_values)),
            "Requested_Alpha_N": int(requested_alpha_n),
            "Alpha_Path_Truncated": bool(len(alpha_values) < requested_alpha_n),
            "Search_Stage": search_stage,
            "Extension_Round": int(extension_round),
            "At_Alpha_Max_Boundary": bool(alpha_index == 0),
            "At_Alpha_Min_Boundary": bool(alpha_index == len(alpha_values) - 1),
            "Superseded_Boundary": False,
            "Validation_Uno_C_10y": validation_uno,
            "Status": status,
        })
    print(
        f"[{feature_set}] DONE {search_stage} | l1_ratio={l1_ratio:g} "
        f"| returned_alphas={len(alpha_values)}/{requested_alpha_n} "
        f"| elapsed={(time.perf_counter()-path_started)/60.0:.2f} min",
        flush=True,
    )
    return rows

def _select_elasticnet_candidate(tuning):
    successful = tuning.loc[np.isfinite(tuning["Validation_Uno_C_10y"])].copy()
    if successful.empty:
        raise RuntimeError("All Elastic-net paths failed")
    best_score = float(successful["Validation_Uno_C_10y"].max())
    candidates = successful.loc[
        successful["Validation_Uno_C_10y"] >= best_score - EN_TIE_TOL
    ].copy()
    # Numerical ties prefer stronger regularization, then the less ridge-like
    # l1 ratio. A point already superseded by a lower-alpha extension is no
    # longer treated as an active search boundary.
    candidates = candidates.sort_values(
        ["Alpha", "L1_Ratio", "At_Alpha_Min_Boundary"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    return candidates.iloc[0]

def load_or_fit_elasticnet(feature_set):
    selected_columns = final_feature_sets[feature_set]
    X_train, X_val, X_test, actual_columns = prepare_X_train_val_test(
        df_train,
        df_val,
        df_test,
        selected_columns,
        scale=True,
    )
    X_train_np = np.ascontiguousarray(X_train.to_numpy(dtype=np.float64, copy=False))
    X_val_np = np.ascontiguousarray(X_val.to_numpy(dtype=np.float64, copy=False))
    X_test_np = np.ascontiguousarray(X_test.to_numpy(dtype=np.float64, copy=False))
    penalty_factor = np.asarray([
        0.0 if column in base_actual_set else 1.0
        for column in actual_columns
    ])
    if not np.any(penalty_factor > 0):
        raise RuntimeError(f"No NMR columns were identified for penalization: {feature_set}")
    if any(column not in set(actual_columns) for column in base_actual_columns):
        raise RuntimeError(f"Core Clinical + PRS columns changed in {feature_set}")

    signature, signature_payload = elasticnet_signature(
        feature_set,
        actual_columns,
        penalty_factor,
    )
    paths = elasticnet_cache_paths(feature_set)
    cached_metadata = None
    tuning_rows = None
    if os.path.exists(paths["risk"]) and os.path.exists(paths["metadata"]):
        with open(paths["metadata"], "r", encoding="utf-8") as handle:
            cached_metadata = json.load(handle)
        if cached_metadata.get("signature") == signature:
            if not bool(cached_metadata.get("selected_at_alpha_min_boundary", False)):
                enforce_elasticnet_boundary_policy(feature_set, cached_metadata)
                with np.load(paths["risk"]) as cached:
                    cached_train_risk = np.asarray(cached["train_risk"], dtype=float).copy()
                    cached_val_risk = np.asarray(cached["val_risk"], dtype=float).copy()
                    cached_test_risk = np.asarray(cached["test_risk"], dtype=float).copy()
                print("Loaded Elastic-net cache:", feature_set, flush=True)
                return {
                    "train_risk": cached_train_risk,
                    "val_risk": cached_val_risk,
                    "test_risk": cached_test_risk,
                    "metadata": cached_metadata,
                    "actual_columns": list(actual_columns),
                    "penalty_factor": penalty_factor,
                    "cache_status": "loaded",
                }
            if os.path.exists(paths["tuning"]):
                cached_tuning = pd.read_csv(paths["tuning"])
                required = {"L1_Ratio", "Alpha", "Validation_Uno_C_10y", "Status"}
                if required.issubset(cached_tuning.columns):
                    if "Search_Stage" not in cached_tuning:
                        cached_tuning["Search_Stage"] = "initial_cached"
                    if "Extension_Round" not in cached_tuning:
                        cached_tuning["Extension_Round"] = 0
                    if "Requested_Alpha_N" not in cached_tuning:
                        cached_tuning["Requested_Alpha_N"] = cached_tuning["Alpha_Path_N"]
                    if "Alpha_Path_Truncated" not in cached_tuning:
                        cached_tuning["Alpha_Path_Truncated"] = False
                    if "Superseded_Boundary" not in cached_tuning:
                        cached_tuning["Superseded_Boundary"] = False
                    tuning_rows = cached_tuning.to_dict("records")
                    print(
                        f"[{feature_set}] RESUME boundary tuning cache; "
                        "extending below the saved minimum alpha",
                        flush=True,
                    )

    if tuning_rows is None:
        tuning_rows = []
        for l1_ratio in EN_L1_RATIOS:
            tuning_rows.extend(_fit_and_score_elasticnet_path(
                feature_set,
                X_train_np,
                X_val_np,
                penalty_factor,
                l1_ratio,
                search_stage="initial",
                extension_round=0,
            ))

    extension_rounds = 0
    while True:
        tuning = pd.DataFrame(tuning_rows)
        best = _select_elasticnet_candidate(tuning)
        if not bool(best["At_Alpha_Min_Boundary"]):
            break
        if extension_rounds >= EN_ALPHA_EXTENSION_MAX_ROUNDS:
            break

        current_alpha = float(best["Alpha"])
        selected_l1_ratio = float(best["L1_Ratio"])
        lower_alpha = max(
            current_alpha * EN_ALPHA_EXTENSION_FACTOR,
            EN_ALPHA_ABSOLUTE_FLOOR,
        )
        if not lower_alpha < current_alpha:
            break
        extension_rounds += 1

        # Once values below this point have been evaluated, the old endpoint is
        # no longer an active minimum boundary even if it remains the optimum.
        for row in tuning_rows:
            if (
                np.isclose(float(row["L1_Ratio"]), selected_l1_ratio)
                and np.isclose(float(row["Alpha"]), current_alpha)
            ):
                row["At_Alpha_Min_Boundary"] = False
                row["Superseded_Boundary"] = True

        extension_alphas = np.geomspace(
            current_alpha,
            lower_alpha,
            EN_ALPHA_EXTENSION_POINTS + 1,
        )[1:]
        tuning_rows.extend(_fit_and_score_elasticnet_path(
            feature_set,
            X_train_np,
            X_val_np,
            penalty_factor,
            selected_l1_ratio,
            search_stage=f"alpha_extension_{extension_rounds}",
            extension_round=extension_rounds,
            explicit_alphas=extension_alphas,
        ))

    tuning = pd.DataFrame(tuning_rows)
    best = _select_elasticnet_candidate(tuning)
    best_l1_ratio = float(best["L1_Ratio"])
    best_alpha = float(best["Alpha"])

    final_model = CoxnetSurvivalAnalysis(
        alphas=[best_alpha],
        l1_ratio=best_l1_ratio,
        penalty_factor=penalty_factor,
        normalize=False,
        max_iter=EN_MAX_ITER,
        tol=1e-7,
        fit_baseline_model=False,
    )
    final_model.fit(X_train_np, y_train)
    train_risk = final_model.predict(X_train_np)
    val_risk = final_model.predict(X_val_np)
    test_risk = final_model.predict(X_test_np)

    coefficients = pd.DataFrame({
        "Feature_Set": feature_set,
        "Variable": actual_columns,
        "Coefficient": np.asarray(final_model.coef_)[:, 0],
        "Penalty_Factor": penalty_factor,
        "Forced_In": penalty_factor == 0,
        "Selected_Alpha": best_alpha,
        "Selected_L1_Ratio": best_l1_ratio,
    })
    metadata = {
        "signature": signature,
        "signature_payload": signature_payload,
        "selected_alpha": best_alpha,
        "selected_l1_ratio": best_l1_ratio,
        "validation_uno_c_10y": float(best["Validation_Uno_C_10y"]),
        "selected_at_alpha_max_boundary": bool(best["At_Alpha_Max_Boundary"]),
        "selected_at_alpha_min_boundary": bool(best["At_Alpha_Min_Boundary"]),
        "selected_at_l1_ratio_min_boundary": bool(np.isclose(best_l1_ratio, min(EN_L1_RATIOS))),
        "selected_at_l1_ratio_max_boundary": bool(np.isclose(best_l1_ratio, max(EN_L1_RATIOS))),
        "accepted_parameter_endpoint": bool(np.isclose(best_l1_ratio, 1.0)),
        "adaptive_alpha_extension_rounds": int(extension_rounds),
        "adaptive_alpha_extension_factor": EN_ALPHA_EXTENSION_FACTOR,
        "minimum_alpha_explored": float(tuning["Alpha"].min()),
        "selected_search_stage": str(best.get("Search_Stage", "unknown")),
        "final_fit_population": "training only",
        "test_used_for_tuning": False,
        "n_nonzero_coefficients": int(np.sum(np.abs(coefficients["Coefficient"]) > 1e-12)),
        "n_penalized_nmr_coefficients": int(np.sum(penalty_factor > 0)),
        "n_forced_core_coefficients": int(np.sum(penalty_factor == 0)),
    }
    np.savez_compressed(
        paths["risk"],
        train_risk=np.asarray(train_risk, dtype=float),
        val_risk=np.asarray(val_risk, dtype=float),
        test_risk=np.asarray(test_risk, dtype=float),
    )
    with open(paths["metadata"], "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, ensure_ascii=False, indent=2)
    tuning.to_csv(paths["tuning"], index=False)
    coefficients.to_csv(paths["coefficients"], index=False)
    print(
        "Fitted and cached Elastic-net:",
        feature_set,
        "alpha=", best_alpha,
        "l1_ratio=", best_l1_ratio,
        "extension_rounds=", extension_rounds,
    )
    enforce_elasticnet_boundary_policy(feature_set, metadata)
    return {
        "train_risk": np.asarray(train_risk, dtype=float),
        "val_risk": np.asarray(val_risk, dtype=float),
        "test_risk": np.asarray(test_risk, dtype=float),
        "metadata": metadata,
        "actual_columns": list(actual_columns),
        "penalty_factor": penalty_factor,
        "cache_status": "fitted_or_extended",
    }

from joblib import Parallel, delayed
_elastic_feature_sets = [feature_set for _, feature_set in RESULT3_REPRESENTATIONS] + [RESULT3_EXPANDED_FEATURE]
_elastic_results = Parallel(
    n_jobs=min(RESULT3_N_JOBS, len(_elastic_feature_sets)), backend="loky", verbose=10, max_nbytes="50M"
)(delayed(load_or_fit_elasticnet)(feature_set) for feature_set in _elastic_feature_sets)
elasticnet_records = dict(zip(_elastic_feature_sets, _elastic_results))
for _feature_name,_record in elasticnet_records.items():
    _meta=_record["metadata"]
    save_risk_record("Elastic-net Cox",_feature_name,_record,_record["actual_columns"],{"selected_alpha":_meta.get("selected_alpha"),"selected_l1_ratio":_meta.get("selected_l1_ratio"),"validation_uno_c_10y":_meta.get("validation_uno_c_10y")},"elasticnet_cox_validation_selected_v2")

# The fixed Clinical + PRS reference is the saved, unpenalized Cox benchmark.
# This is appropriate because every added NMR model forces the same core block in.
elasticnet_records[RESULT3_BASE_FEATURE] = {
    **result3_cox_record(RESULT3_BASE_FEATURE),
    "metadata": {
        "reference_model": "saved unpenalized Cox PH Clinical + PRS",
        "reason": "No NMR variables exist in the reference block to penalize",
    },
    "cache_status": "shared_reference",
}

elasticnet_manifest = pd.DataFrame([
    {
        "Feature_Set": feature_set,
        "Cache_Status": record["cache_status"],
        "Selected_Alpha": record["metadata"].get("selected_alpha", np.nan),
        "Selected_L1_Ratio": record["metadata"].get("selected_l1_ratio", np.nan),
        "Validation_Uno_C_10y": record["metadata"].get("validation_uno_c_10y", np.nan),
        "Alpha_Min_Boundary": record["metadata"].get("selected_at_alpha_min_boundary", False),
        "Alpha_Max_Boundary": record["metadata"].get("selected_at_alpha_max_boundary", False),
        "L1_Ratio_Min_Boundary": record["metadata"].get("selected_at_l1_ratio_min_boundary", False),
        "L1_Ratio_Max_Boundary": record["metadata"].get("selected_at_l1_ratio_max_boundary", False),
        "Accepted_Parameter_Endpoint": record["metadata"].get("accepted_parameter_endpoint", False),
        "Final_Fit_Population": record["metadata"].get("final_fit_population", "saved reference"),
        "Test_Used_For_Tuning": record["metadata"].get("test_used_for_tuning", False),
    }
    for feature_set, record in elasticnet_records.items()
])
elasticnet_manifest.to_csv(
    os.path.join(RESULT3_SOURCE_DIR, "elasticnet_model_selection_manifest.csv"),
    index=False,
)
ipy_display(elasticnet_manifest)


In [ ]:
ALGORITHMS = ["Cox PH", "Elastic-net Cox"]

def result3_risk_record(algorithm, feature_set):
    if algorithm == "Cox PH":
        return result3_cox_record(feature_set)
    if algorithm == "Elastic-net Cox":
        return elasticnet_records[feature_set]
    raise KeyError(algorithm)

result3_calibrators = {}
result3_predictions = {}
result3_ibs_objects = {}
for algorithm in ALGORITHMS:
    algorithm_features = RESULT3_FEATURES + (
        [RESULT3_EXPANDED_FEATURE]
        if algorithm in RESULT3_EXPANDED_ALGORITHMS else []
    )
    for feature_set in algorithm_features:
        record = result3_risk_record(algorithm, feature_set)
        calibrator = fit_absolute_risk_calibrator(
            df_train,
            record["train_risk"],
            label=f"{feature_set} | {algorithm}",
        )
        result3_calibrators[(algorithm, feature_set)] = calibrator
        for horizon in HORIZONS:
            result3_predictions[(algorithm, feature_set, horizon)] = predict_absolute_risk(
                calibrator,
                record["test_risk"],
                horizon,
            )
            result3_ibs_objects[(algorithm, feature_set, horizon)] = subject_level_ipcw_ibs(
                df_train,
                df_test,
                calibrator,
                record["test_risk"],
                horizon,
            )


In [ ]:
# Result 3 CPU parallel configuration

import os
import time
import numpy as np
import pandas as pd

from joblib import Parallel, delayed

# 1. Detect CPUs actually available to this Jupyter process

try:
    RESULT3_VISIBLE_CPUS = len(os.sched_getaffinity(0))
except Exception:
    RESULT3_VISIBLE_CPUS = os.cpu_count() or 1

# RESULT3_N_JOBS was defined and validated once in the Elastic-net
# configuration cell, so fitting and downstream metric pools share one budget.
print(
    f"Visible CPUs: {RESULT3_VISIBLE_CPUS} | "
    f"Result3 maximum workers: {RESULT3_N_JOBS}"
)

# 2. Metric calculations only need Time + Event
#    Do not pass the entire 100+ column UKB dataframe to workers.

result3_train_surv = (
    df_train[["Time", "Event"]]
    .reset_index(drop=True)
    .copy()
)

result3_test_surv = (
    df_test[["Time", "Event"]]
    .reset_index(drop=True)
    .copy()
)

# 3. Worker: Uno IPCW C-index

def _result3_worker_uno(
    train_frame,
    test_frame,
    risk_new,
    risk_reference,
    horizon,
    seed,
    section,
    metadata,
):
    row = paired_bootstrap_uno(
        train_frame,
        test_frame,
        np.asarray(risk_new, dtype=float),
        np.asarray(risk_reference, dtype=float),
        horizon,
        seed=seed,
    )

    return section, {
        **metadata,
        **row,
    }

# 4. Worker: Harrell C

def _result3_worker_harrell(
    test_frame,
    risk_new,
    risk_reference,
    seed,
    metadata,
):
    row = paired_bootstrap_harrell(
        test_frame,
        np.asarray(risk_new, dtype=float),
        np.asarray(risk_reference, dtype=float),
        seed=seed,
    )

    return {
        **metadata,
        **row,
    }

# 5. Worker: IPCW IBS

def _result3_worker_ibs(
    new_ibs,
    reference_ibs,
    seed,
    metadata,
):
    row = paired_bootstrap_delta_ibs(
        new_ibs,
        reference_ibs,
        seed=seed,
    )

    return {
        **metadata,
        **row,
    }

# 6. Worker: IPCW NRI / IDI

def _result3_worker_nri_idi(
    test_frame,
    pred_reference,
    pred_new,
    horizon,
    cutpoints,
    seed,
    metadata,
):
    row = bootstrap_ipcw_nri_idi(
        test_frame,
        np.asarray(pred_reference, dtype=float),
        np.asarray(pred_new, dtype=float),
        horizon,
        cutpoints=cutpoints,
        seed=seed,
    )

    return {
        **metadata,
        **row,
    }

def _result3_parallel(function, jobs, label):
    """
    Run independent Result3 jobs with process-based parallelism.
    Statistical functions themselves are unchanged.
    """
    if len(jobs) == 0:
        return []

    n_jobs = min(RESULT3_N_JOBS, len(jobs))

    print()
    print("=" * 70)
    print(f"{label}")
    print(f"Tasks   : {len(jobs)}")
    print(f"Workers : {n_jobs}")
    print("=" * 70)

    start = time.time()

    results = Parallel(
        n_jobs=n_jobs,
        backend="loky",
        verbose=10,
        max_nbytes="50M",
    )(
        delayed(function)(*job)
        for job in jobs
    )

    elapsed = (time.time() - start) / 60.0

    print(
        f"{label} complete | "
        f"{elapsed:.1f} min"
    )

    return results


In [ ]:
# Figure 3 source data.

# Figure 3A

result3_scale = pd.DataFrame([
    {
        "Representation": representation_display,
        "Feature_Set": feature_set,
        "NMR_Predictor_N": int(
            len(
                set(final_feature_sets[feature_set])
                - set(final_feature_sets[RESULT3_BASE_FEATURE])
            )
        ),
    }
    for representation_display, feature_set in RESULT3_REPRESENTATIONS
])

result3_scale.to_csv(
    os.path.join(
        RESULT3_SOURCE_DIR,
        "figure3A_representation_scale.csv",
    ),
    index=False,
)

print("Figure 3A complete.")

# Figure 3B + Figure 3C
#
# Combine all Uno IPCW tasks into ONE parallel pool:
#
# Figure 3B:
#   2 algorithms × 3 representations × 2 horizons = 12 tasks
#
# Figure 3C:
#   3 representations × 2 horizons = 6 tasks
#
# Total = 18 independent bootstrap tasks

uno_jobs = []

# Figure 3B jobs

for algorithm_index, algorithm in enumerate(ALGORITHMS):

    for representation_index, (
        representation_display,
        feature_set,
    ) in enumerate(RESULT3_REPRESENTATIONS):

        for horizon_index, horizon in enumerate(HORIZONS):

            seed = (
                BOOTSTRAP_SEED
                + 2000
                + algorithm_index * 100
                + representation_index * 10
                + horizon_index
            )

            metadata = {
                "Algorithm": algorithm,
                "Representation": representation_display,
                "Feature_Set": feature_set,
                "Reference_Feature_Set": RESULT3_BASE_FEATURE,
                "Horizon_Years": horizon,
                "Split": "test",
            }

            uno_jobs.append(
                (
                    result3_train_surv,
                    result3_test_surv,

                    result3_risk_record(
                        algorithm,
                        feature_set,
                    )["test_risk"],

                    result3_risk_record(
                        algorithm,
                        RESULT3_BASE_FEATURE,
                    )["test_risk"],

                    horizon,
                    seed,
                    "B",
                    metadata,
                )
            )

# Figure 3C jobs

for representation_index, (
    representation_display,
    feature_set,
) in enumerate(RESULT3_REPRESENTATIONS):

    for horizon_index, horizon in enumerate(HORIZONS):

        seed = (
            BOOTSTRAP_SEED
            + 2200
            + representation_index * 10
            + horizon_index
        )

        metadata = {
            "Comparison": "Elastic-net Cox minus Cox PH",
            "Representation": representation_display,
            "Feature_Set": feature_set,
            "Horizon_Years": horizon,
            "Split": "test",
        }

        uno_jobs.append(
            (
                result3_train_surv,
                result3_test_surv,

                result3_risk_record(
                    "Elastic-net Cox",
                    feature_set,
                )["test_risk"],

                result3_risk_record(
                    "Cox PH",
                    feature_set,
                )["test_risk"],

                horizon,
                seed,
                "C",
                metadata,
            )
        )

uno_results = _result3_parallel(
    _result3_worker_uno,
    uno_jobs,
    "Figure 3B + 3C: Uno IPCW bootstrap",
)

# Split B and C results again
result3_b = pd.DataFrame([
    row
    for section, row in uno_results
    if section == "B"
])

result3_c = pd.DataFrame([
    row
    for section, row in uno_results
    if section == "C"
])

result3_b.to_csv(
    os.path.join(
        RESULT3_SOURCE_DIR,
        "figure3B_delta_uno_cindex_vs_clinical_prs.csv",
    ),
    index=False,
)

result3_c.to_csv(
    os.path.join(
        RESULT3_SOURCE_DIR,
        "figure3C_elasticnet_minus_cox_delta_uno_cindex.csv",
    ),
    index=False,
)

print("Figure 3B complete.")
print("Figure 3C complete.")

# Supplementary Harrell C

harrell_jobs = []

for representation_index, (
    representation_display,
    feature_set,
) in enumerate(RESULT3_REPRESENTATIONS):

    seed = (
        BOOTSTRAP_SEED
        + 2250
        + representation_index
    )

    metadata = {
        "Comparison": "Elastic-net Cox minus Cox PH",
        "Representation": representation_display,
        "Feature_Set": feature_set,
        "Split": "test",
    }

    harrell_jobs.append(
        (
            result3_test_surv,

            result3_risk_record(
                "Elastic-net Cox",
                feature_set,
            )["test_risk"],

            result3_risk_record(
                "Cox PH",
                feature_set,
            )["test_risk"],

            seed,
            metadata,
        )
    )

result3_harrell_rows = _result3_parallel(
    _result3_worker_harrell,
    harrell_jobs,
    "Supplementary Harrell C bootstrap",
)

result3_harrell = pd.DataFrame(
    result3_harrell_rows
)

result3_harrell.to_csv(
    os.path.join(
        RESULT3_SOURCE_DIR,
        "supplement_elasticnet_minus_cox_delta_harrell_cindex.csv",
    ),
    index=False,
)

print("Supplementary Harrell C complete.")

# Figure 3D: IPCW IBS

ibs_jobs = []

for algorithm_index, algorithm in enumerate(ALGORITHMS):

    for representation_index, (
        representation_display,
        feature_set,
    ) in enumerate(RESULT3_REPRESENTATIONS):

        for horizon_index, horizon in enumerate(HORIZONS):

            seed = (
                BOOTSTRAP_SEED
                + 2300
                + algorithm_index * 100
                + representation_index * 10
                + horizon_index
            )

            metadata = {
                "Algorithm": algorithm,
                "Representation": representation_display,
                "Feature_Set": feature_set,
                "Reference_Feature_Set": RESULT3_BASE_FEATURE,
                "Horizon_Years": horizon,
                "Split": "test",
                "Delta_Definition":
                    "IBS_model_minus_IBS_clinical_prs; lower_is_better",
            }

            ibs_jobs.append(
                (
                    result3_ibs_objects[
                        (
                            algorithm,
                            feature_set,
                            horizon,
                        )
                    ],

                    result3_ibs_objects[
                        (
                            algorithm,
                            RESULT3_BASE_FEATURE,
                            horizon,
                        )
                    ],

                    seed,
                    metadata,
                )
            )

result3_d_rows = _result3_parallel(
    _result3_worker_ibs,
    ibs_jobs,
    "Figure 3D: IPCW IBS bootstrap",
)

result3_d = pd.DataFrame(
    result3_d_rows
)

result3_d.to_csv(
    os.path.join(
        RESULT3_SOURCE_DIR,
        "figure3D_delta_ipcw_ibs.csv",
    ),
    index=False,
)

print("Figure 3D complete.")

# Figure 3E: IPCW NRI / IDI
#
# This is one of the slowest parts because the censoring KM
# is refitted inside every test bootstrap.
#
# 2 algorithms × 3 representations × 2 horizons
# = 12 independent tasks

nri_jobs = []

for algorithm_index, algorithm in enumerate(ALGORITHMS):

    for representation_index, (
        representation_display,
        feature_set,
    ) in enumerate(RESULT3_REPRESENTATIONS):

        for horizon_index, horizon in enumerate(HORIZONS):

            seed = (
                BOOTSTRAP_SEED
                + 2400
                + algorithm_index * 100
                + representation_index * 10
                + horizon_index
            )

            metadata = {
                "Algorithm": algorithm,
                "Representation": representation_display,
                "Feature_Set": feature_set,
                "Reference_Feature_Set":
                    RESULT3_BASE_FEATURE,
                "Horizon_Years": horizon,
                "Split": "test",
                "NRI_Cutpoints":
                    ";".join(
                        str(value)
                        for value in NRI_CUTPOINTS
                    ),
                "Censoring_Method":
                    "IPCW; censoring KM refitted within each test bootstrap",
            }

            nri_jobs.append(
                (
                    result3_test_surv,

                    result3_predictions[
                        (
                            algorithm,
                            RESULT3_BASE_FEATURE,
                            horizon,
                        )
                    ],

                    result3_predictions[
                        (
                            algorithm,
                            feature_set,
                            horizon,
                        )
                    ],

                    horizon,
                    NRI_CUTPOINTS,
                    seed,
                    metadata,
                )
            )

result3_e_rows = _result3_parallel(
    _result3_worker_nri_idi,
    nri_jobs,
    "Figure 3E: IPCW NRI / IDI bootstrap",
)

result3_e = pd.DataFrame(
    result3_e_rows
)

result3_e.to_csv(
    os.path.join(
        RESULT3_SOURCE_DIR,
        "figure3E_ipcw_nri_idi.csv",
    ),
    index=False,
)

print("Figure 3E complete.")

# Compact long-format summary for manuscript table preparation.
summary_parts = []

def _add_result3_summary(frame, metric, estimate, lower, upper):
    part = frame[["Algorithm", "Representation", "Feature_Set", "Horizon_Years", "Split"]].copy()
    part["Metric"] = metric
    part["Estimate"] = frame[estimate].astype(float)
    part["Lower95"] = frame[lower].astype(float)
    part["Upper95"] = frame[upper].astype(float)
    summary_parts.append(part)

_add_result3_summary(
    result3_b,
    "Delta Uno IPCW C-index vs Clinical + PRS",
    "Delta_Uno_C", "Delta_Uno_C_Lower95", "Delta_Uno_C_Upper95",
)
_add_result3_summary(
    result3_d,
    "Delta IPCW IBS vs Clinical + PRS",
    "Delta_IBS", "Delta_IBS_Lower95", "Delta_IBS_Upper95",
)
_add_result3_summary(
    result3_e,
    "Categorical NRI vs Clinical + PRS",
    "NRI_Categorical", "NRI_Categorical_Lower95", "NRI_Categorical_Upper95",
)
_add_result3_summary(
    result3_e,
    "IDI vs Clinical + PRS",
    "IDI", "IDI_Lower95", "IDI_Upper95",
)
result3_summary = pd.concat(summary_parts, ignore_index=True)
result3_summary.to_csv(
    os.path.join(RESULT3_SOURCE_DIR, "result3_summary_long.csv"),
    index=False,
)

# Figure 3F: Calibration
#
# This section is not the major bottleneck.
# Keep it serial to avoid unnecessary process overhead.

calibration_parts = []

for algorithm in ALGORITHMS:

    for representation_display, feature_set in (
        [("Clinical + PRS", RESULT3_BASE_FEATURE)]
        + RESULT3_REPRESENTATIONS
    ):

        for horizon in HORIZONS:

            calibration = calibration_by_quantile(
                result3_test_surv,
                result3_predictions[
                    (
                        algorithm,
                        feature_set,
                        horizon,
                    )
                ],
                horizon,
            )

            calibration["Algorithm"] = algorithm
            calibration["Representation"] = representation_display
            calibration["Feature_Set"] = feature_set
            calibration["Horizon_Years"] = horizon
            calibration["Split"] = "test"

            calibration_parts.append(
                calibration
            )

result3_f = pd.concat(
    calibration_parts,
    ignore_index=True,
)

result3_f.to_csv(
    os.path.join(
        RESULT3_SOURCE_DIR,
        "figure3F_calibration.csv",
    ),
    index=False,
)

print("Figure 3F complete.")

# Figure 3G: Survival DCA
#
# dcurves already performs the survival-DCA calculation.
# No additional IPCW is added here.

dca_parts = []

for algorithm in ALGORITHMS:

    for horizon in HORIZONS:

        predictions = {

            "Clinical + PRS":
                result3_predictions[
                    (
                        algorithm,
                        RESULT3_BASE_FEATURE,
                        horizon,
                    )
                ],

            **{
                representation_display:
                    result3_predictions[
                        (
                            algorithm,
                            feature_set,
                            horizon,
                        )
                    ]

                for representation_display, feature_set
                in RESULT3_REPRESENTATIONS
            },
        }

        dca_frame = survival_dca(
            result3_test_surv,
            predictions,
            horizon,
        )

        dca_frame["Algorithm"] = algorithm

        dca_parts.append(
            dca_frame
        )

result3_g = pd.concat(
    dca_parts,
    ignore_index=True,
)

result3_g.to_csv(
    os.path.join(
        RESULT3_SOURCE_DIR,
        "figure3G_survival_dca.csv",
    ),
    index=False,
)

print("Figure 3G complete.")

# Finished

print()
print("=" * 70)
print("RESULT 3 SOURCE DATA COMPLETE")
print("=" * 70)


In [ ]:
from IPython.display import display as ipy_display

ipy_display(result3_b)
ipy_display(result3_c)
ipy_display(result3_d)
ipy_display(result3_e)


In [ ]:
# Figure 3A: representation scale.
fig, ax = plt.subplots(figsize=(7.4, 4.3))
CNS["red"] = "#C44E52"
bar_colors = [CNS["green"], CNS["purple"], CNS["red"]]
bars = ax.bar(
    result3_scale["Representation"],
    result3_scale["NMR_Predictor_N"],
    color=bar_colors,
    width=0.65,
)
for bar, value in zip(bars, result3_scale["NMR_Predictor_N"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(int(value)), ha="center", va="bottom")
ax.set_ylabel("Number of penalized NMR predictors")
ax.set_title("Figure 3A. NMR representation scale", loc="left", fontweight="bold")
ax.grid(axis="y", color=CNS["light_gray"], lw=0.8)
clean_axes(ax)
fig.tight_layout()
save_figure(fig, RESULT3_FIGURE_DIR, "figure3A_nmr_representation_scale")
plt.show()


In [ ]:
# Figure 3B: primary Cox and Elastic-net sensitivity are separate files.
horizon_style = {
    5.0: (CNS["teal"], "o", -0.11, "5 years"),
    10.0: (CNS["blue"], "s", 0.11, "10 years"),
}
representation_order = [label for label, _ in RESULT3_REPRESENTATIONS]
for algorithm in ALGORITHMS:
    data_algorithm = result3_b.loc[result3_b["Algorithm"].astype(str) == algorithm].copy()
    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    y = np.arange(len(representation_order))
    for horizon, (color, marker, offset, legend_label) in horizon_style.items():
        data = data_algorithm.loc[np.isclose(data_algorithm["Horizon_Years"], horizon)].copy()
        data["Order"] = data["Representation"].map({label: i for i, label in enumerate(representation_order)})
        data = data.sort_values("Order")
        estimate = data["Delta_Uno_C"].to_numpy(float)
        lower = data["Delta_Uno_C_Lower95"].to_numpy(float)
        upper = data["Delta_Uno_C_Upper95"].to_numpy(float)
        ax.errorbar(estimate, y + offset, xerr=[np.maximum(0, estimate-lower), np.maximum(0, upper-estimate)],
                    fmt=marker, color=color, ecolor=color, capsize=4, markersize=7, label=legend_label)
    ax.axvline(0, color=CNS["gray"], lw=1)
    ax.set_yticks(y); ax.set_yticklabels(representation_order); ax.invert_yaxis()
    ax.set_xlabel("Delta Uno IPCW C-index vs Clinical + PRS (95% CI)")
    role = "primary Cox analysis" if algorithm == "Cox PH" else "Elastic-net sensitivity"
    ax.set_title(f"Figure 3B. {role}", loc="left", fontweight="bold")
    ax.grid(axis="x", color=CNS["light_gray"], lw=0.8); ax.legend(frameon=False)
    clean_axes(ax); fig.tight_layout()
    safe = "cox_primary" if algorithm == "Cox PH" else "elasticnet_sensitivity"
    save_figure(fig, RESULT3_FIGURE_DIR, f"figure3B_{safe}_delta_uno_cindex")
    plt.show()


In [ ]:
# Figure 3C: direct Elastic-net minus ordinary Cox comparison.
fig, ax = plt.subplots(figsize=(8.2, 4.3))
y = np.arange(len(RESULT3_REPRESENTATIONS))
representation_order = [representation_display for representation_display, _ in RESULT3_REPRESENTATIONS]
for horizon, (color, marker, offset, legend_label) in horizon_style.items():
    data = result3_c.loc[np.isclose(result3_c["Horizon_Years"], horizon)].copy()
    data["Order"] = data["Representation"].map({label: index for index, label in enumerate(representation_order)})
    data = data.sort_values("Order")
    estimate = data["Delta_Uno_C"].to_numpy(dtype=float)
    lower = data["Delta_Uno_C_Lower95"].to_numpy(dtype=float)
    upper = data["Delta_Uno_C_Upper95"].to_numpy(dtype=float)
    ax.errorbar(
        estimate,
        y + offset,
        xerr=[np.maximum(0, estimate - lower), np.maximum(0, upper - estimate)],
        fmt=marker,
        color=color,
        ecolor=color,
        capsize=3,
        markersize=6,
        label=legend_label,
    )
ax.axvline(0, color=CNS["gray"], lw=1)
ax.set_yticks(y)
ax.set_yticklabels(representation_order)
ax.invert_yaxis()
ax.set_xlabel("Delta Uno IPCW C-index: Elastic-net minus Cox PH (95% CI)")
ax.set_title("Figure 3C. Direct modeling-strategy comparison", loc="left", fontweight="bold")
ax.grid(axis="x", color=CNS["light_gray"], lw=0.8)
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()
save_figure(fig, RESULT3_FIGURE_DIR, "figure3C_elasticnet_minus_cox_delta_uno_cindex")
plt.show()


In [ ]:
# Figure 3D: IPCW Delta IBS, one file per analysis role.
representation_order = [label for label, _ in RESULT3_REPRESENTATIONS]
scale = 1e5 if np.nanmax(np.abs(result3_d[["Delta_IBS", "Delta_IBS_Lower95", "Delta_IBS_Upper95"]])) < 1e-3 else 1.0
scale_label = " (x10^-5)" if scale == 1e5 else ""
for algorithm in ALGORITHMS:
    data_algorithm = result3_d.loc[result3_d["Algorithm"].astype(str) == algorithm].copy()
    fig, ax = plt.subplots(figsize=(9.2, 5.4)); y = np.arange(len(representation_order))
    for horizon, (color, marker, offset, legend_label) in horizon_style.items():
        data = data_algorithm.loc[np.isclose(data_algorithm["Horizon_Years"], horizon)].copy()
        data["Order"] = data["Representation"].map({label: i for i, label in enumerate(representation_order)})
        data = data.sort_values("Order")
        estimate = data["Delta_IBS"].to_numpy(float) * scale
        lower = data["Delta_IBS_Lower95"].to_numpy(float) * scale
        upper = data["Delta_IBS_Upper95"].to_numpy(float) * scale
        ax.errorbar(estimate, y + offset, xerr=[np.maximum(0, estimate-lower), np.maximum(0, upper-estimate)],
                    fmt=marker, color=color, ecolor=color, capsize=4, markersize=7, label=legend_label)
    ax.axvline(0, color=CNS["gray"], lw=1)
    ax.set_yticks(y); ax.set_yticklabels(representation_order); ax.invert_yaxis()
    ax.set_xlabel(f"Delta IPCW IBS vs Clinical + PRS{scale_label} (lower is better)")
    role = "primary Cox analysis" if algorithm == "Cox PH" else "Elastic-net sensitivity"
    ax.set_title(f"Figure 3D. {role}", loc="left", fontweight="bold")
    ax.grid(axis="x", color=CNS["light_gray"], lw=0.8); ax.legend(frameon=False)
    clean_axes(ax); fig.tight_layout()
    safe = "cox_primary" if algorithm == "Cox PH" else "elasticnet_sensitivity"
    save_figure(fig, RESULT3_FIGURE_DIR, f"figure3D_{safe}_delta_ipcw_ibs")
    plt.show()


In [ ]:
# Figure 3E: NRI/IDI separated by algorithm and horizon.
representation_order = [label for label, _ in RESULT3_REPRESENTATIONS]
for algorithm in ALGORITHMS:
    for horizon in HORIZONS:
        data = result3_e.loc[
            (result3_e["Algorithm"].astype(str) == algorithm) & np.isclose(result3_e["Horizon_Years"], horizon)
        ].copy()
        data["Order"] = data["Representation"].map({label: i for i, label in enumerate(representation_order)})
        data = data.sort_values("Order")
        fig, ax = plt.subplots(figsize=(9.2, 5.4)); y = np.arange(len(data))
        for metric, low, high, color, marker, offset, label in [
            ("NRI_Categorical", "NRI_Categorical_Lower95", "NRI_Categorical_Upper95", CNS["green"], "o", -0.12, "Categorical NRI"),
            ("IDI", "IDI_Lower95", "IDI_Upper95", CNS["orange"], "s", 0.12, "IDI"),
        ]:
            estimate = data[metric].to_numpy(float); lower = data[low].to_numpy(float); upper = data[high].to_numpy(float)
            ax.errorbar(estimate, y + offset, xerr=[np.maximum(0, estimate-lower), np.maximum(0, upper-estimate)],
                        fmt=marker, color=color, ecolor=color, capsize=4, markersize=7, label=label)
        ax.axvline(0, color=CNS["gray"], lw=1)
        ax.set_yticks(y); ax.set_yticklabels(data["Representation"]); ax.invert_yaxis()
        ax.set_xlabel("IPCW estimate vs Clinical + PRS (95% CI)")
        role = "Cox primary" if algorithm == "Cox PH" else "Elastic-net sensitivity"
        ax.set_title(f"Figure 3E. {role}: {horizon:g}-year reclassification", loc="left", fontweight="bold")
        ax.grid(axis="x", color=CNS["light_gray"], lw=0.8); ax.legend(frameon=False)
        clean_axes(ax); fig.tight_layout()
        safe = "cox_primary" if algorithm == "Cox PH" else "elasticnet_sensitivity"
        save_figure(fig, RESULT3_FIGURE_DIR, f"figure3E_{safe}_{int(horizon)}y_ipcw_nri_idi")
        plt.show()


In [ ]:
# Figure 3F: calibration figures are separated by algorithm and horizon.
calibration_colors = {
    "Clinical + PRS": CNS["black"],
    "NMR_score": CNS["green"],
    "Spearman-pruned NMR": CNS["purple"],
    "Raw106 NMR": CNS["red"],
}
for algorithm in ALGORITHMS:
    for horizon in HORIZONS:
        data = result3_f.loc[
            (result3_f["Algorithm"].astype(str) == algorithm)
            & np.isclose(result3_f["Horizon_Years"], horizon)
        ].copy()
        fig, ax = plt.subplots(figsize=(6.4, 5.4))
        maximum = float(np.nanmax(data[["Predicted_Risk", "Observed_KM_Risk"]].to_numpy()))
        ax.plot([0, maximum], [0, maximum], color=CNS["gray"], linestyle="--", lw=1.2, label="Ideal")
        for representation, group in data.groupby("Representation"):
            group = group.sort_values("Predicted_Risk")
            ax.plot(
                group["Predicted_Risk"],
                group["Observed_KM_Risk"],
                marker="o",
                markersize=4,
                linewidth=1.8,
                color=calibration_colors.get(representation, CNS["gray"]),
                label=representation,
            )
        ax.set_xlabel(f"Predicted {horizon:g}-year CHD risk")
        ax.set_ylabel(f"Observed {horizon:g}-year CHD risk (KM)")
        ax.set_title(f"Figure 3F. {algorithm}: {horizon:g}-year calibration", loc="left", fontweight="bold")
        ax.grid(color=CNS["light_gray"], lw=0.8)
        ax.legend(frameon=False, fontsize=11.0)
        clean_axes(ax)
        fig.tight_layout()
        safe_algorithm = "elasticnet" if algorithm == "Elastic-net Cox" else "cox"
        save_figure(fig, RESULT3_FIGURE_DIR, f"figure3F_{safe_algorithm}_{int(horizon)}y_calibration")
        plt.show()


In [ ]:
# Figure 3G: standard survival DCA, separated by algorithm and horizon.
dca_colors = {
    "Clinical + PRS": CNS["black"],
    "NMR_score": CNS["green"],
    "Spearman-pruned NMR": CNS["purple"],
    "Raw106 NMR": CNS["red"],
    "Treat all": CNS["gray"],
    "Treat none": CNS["gray"],
}
for algorithm in ALGORITHMS:
    for horizon in HORIZONS:
        data = result3_g.loc[
            (result3_g["Algorithm"].astype(str) == algorithm)
            & np.isclose(result3_g["Horizon_Years"], horizon)
        ].copy()
        safe_algorithm = "elasticnet" if algorithm == "Elastic-net Cox" else "cox"
        draw_dca(
            data,
            title=f"Figure 3G. {algorithm}: {horizon:g}-year survival DCA",
            output_dir=RESULT3_FIGURE_DIR,
            stem=f"figure3G_{safe_algorithm}_{int(horizon)}y_survival_dca",
            colors=dca_colors,
        )


In [ ]:
# Supplementary Harrell C direct comparison.
draw_forest(
    result3_harrell,
    estimate="Delta_Harrell_C",
    lower="Delta_Harrell_C_Lower95",
    upper="Delta_Harrell_C_Upper95",
    label="Representation",
    title="Supplementary Figure. Elastic-net versus Cox PH using Harrell C",
    xlabel="Paired delta Harrell C-index (95% CI)",
    output_dir=RESULT3_FIGURE_DIR,
    stem="supplement_elasticnet_minus_cox_delta_harrell_cindex",
    color=CNS["purple"],
)

print("Result 3 complete.")
print("Ordinary Cox models were fitted or resumed from signed caches in this notebook.")
print("Elastic-net models were loaded from cache or fit once using validation-only tuning.")
print("Figures:", RESULT3_FIGURE_DIR)
print("Source data:", RESULT3_SOURCE_DIR)


In [ ]:
# Supplementary expanded-panel sensitivity; never enters the three-representation primary figures.
expanded_uno_jobs = []
for algorithm_index, algorithm in enumerate(RESULT3_EXPANDED_ALGORITHMS):
    for horizon_index, horizon in enumerate(HORIZONS):
        metadata = {
            "Algorithm": algorithm,
            "Representation": RESULT3_EXPANDED_LABEL,
            "Feature_Set": RESULT3_EXPANDED_FEATURE,
            "Reference_Feature_Set": RESULT3_BASE_FEATURE,
            "Analysis_Role": "expanded_panel_sensitivity",
            "Horizon_Years": horizon,
            "Split": "test",
        }
        expanded_uno_jobs.append((
            result3_train_surv, result3_test_surv,
            result3_risk_record(algorithm, RESULT3_EXPANDED_FEATURE)["test_risk"],
            result3_risk_record(algorithm, RESULT3_BASE_FEATURE)["test_risk"],
            horizon, BOOTSTRAP_SEED + 2800 + algorithm_index * 10 + horizon_index,
            "expanded", metadata,
        ))

expanded_uno_result = _result3_parallel(
    _result3_worker_uno, expanded_uno_jobs,
    "Supplementary Full249 expanded-panel Uno IPCW bootstrap",
)
result3_expanded_uno = pd.DataFrame([row for _, row in expanded_uno_result])
atomic_write_csv(
    result3_expanded_uno,
    os.path.join(RESULT3_SOURCE_DIR, "supplement_result3_full249_expanded_panel_delta_uno.csv"),
    index=False,
)

expanded_ibs_rows = []
expanded_calibration_parts = []
expanded_dca_parts = []
for algorithm_index, algorithm in enumerate(RESULT3_EXPANDED_ALGORITHMS):
    for horizon_index, horizon in enumerate(HORIZONS):
        ibs_row = paired_bootstrap_delta_ibs(
            result3_ibs_objects[(algorithm, RESULT3_EXPANDED_FEATURE, horizon)],
            result3_ibs_objects[(algorithm, RESULT3_BASE_FEATURE, horizon)],
            seed=BOOTSTRAP_SEED + 2900 + algorithm_index * 10 + horizon_index,
        )
        expanded_ibs_rows.append({
            "Algorithm": algorithm, "Representation": RESULT3_EXPANDED_LABEL,
            "Feature_Set": RESULT3_EXPANDED_FEATURE,
            "Reference_Feature_Set": RESULT3_BASE_FEATURE,
            "Analysis_Role": "expanded_panel_sensitivity",
            "Horizon_Years": horizon, "Split": "test", **ibs_row,
        })
        for label, feature_set in [("Clinical + PRS", RESULT3_BASE_FEATURE), (RESULT3_EXPANDED_LABEL, RESULT3_EXPANDED_FEATURE)]:
            grouped = calibration_by_quantile(
                result3_test_surv, result3_predictions[(algorithm, feature_set, horizon)], horizon
            )
            grouped["Algorithm"] = algorithm; grouped["Representation"] = label
            grouped["Feature_Set"] = feature_set; grouped["Horizon_Years"] = horizon
            grouped["Analysis_Role"] = "expanded_panel_sensitivity"; grouped["Split"] = "test"
            expanded_calibration_parts.append(grouped)
        dca = survival_dca(
            result3_test_surv,
            {
                "Clinical + PRS": result3_predictions[(algorithm, RESULT3_BASE_FEATURE, horizon)],
                RESULT3_EXPANDED_LABEL: result3_predictions[(algorithm, RESULT3_EXPANDED_FEATURE, horizon)],
            },
            horizon,
        )
        dca["Algorithm"] = algorithm; dca["Analysis_Role"] = "expanded_panel_sensitivity"
        expanded_dca_parts.append(dca)

result3_expanded_ibs = pd.DataFrame(expanded_ibs_rows)
result3_expanded_calibration = pd.concat(expanded_calibration_parts, ignore_index=True)
result3_expanded_dca = pd.concat(expanded_dca_parts, ignore_index=True)
atomic_write_csv(result3_expanded_ibs, os.path.join(RESULT3_SOURCE_DIR, "supplement_result3_full249_expanded_panel_ipcw_ibs.csv"), index=False)
atomic_write_csv(result3_expanded_calibration, os.path.join(RESULT3_SOURCE_DIR, "supplement_result3_full249_expanded_panel_calibration.csv"), index=False)
atomic_write_csv(result3_expanded_dca, os.path.join(RESULT3_SOURCE_DIR, "supplement_result3_full249_expanded_panel_survival_dca.csv"), index=False)

expanded_role_audit = pd.DataFrame([{
    "Feature_Set": RESULT3_EXPANDED_FEATURE,
    "Representation": RESULT3_EXPANDED_LABEL,
    "Analysis_Role": "expanded_panel_sensitivity",
    "Included_In_Primary_Result3_Figures": False,
    "Included_In_Result4": False,
    "Models_Available": " | ".join(RESULT3_EXPANDED_ALGORITHMS),
    "Cox_Status": EXPANDED_COX_STATUS,
    "Cox_Error": EXPANDED_COX_ERROR,
    "Cox_Failure_Blocks_Primary_Result3": False,
    "Elasticnet_Tuning": "validation Uno IPCW C at 10 years; test untouched",
}])
atomic_write_csv(expanded_role_audit, os.path.join(RESULT3_SOURCE_DIR, "supplement_result3_full249_expanded_panel_role_audit.csv"), index=False)

for algorithm in RESULT3_EXPANDED_ALGORITHMS:
    forest = result3_expanded_uno.loc[result3_expanded_uno["Algorithm"].eq(algorithm)].copy()
    forest["Horizon_Label"] = forest["Horizon_Years"].map(lambda value: f"{value:g} years")
    draw_forest(
        forest, estimate="Delta_Uno_C", lower="Delta_Uno_C_Lower95", upper="Delta_Uno_C_Upper95",
        label="Horizon_Label",
        title=f"Supplementary Figure. Full249 expanded panel: {algorithm}",
        xlabel="Delta Uno IPCW C-index vs Clinical + PRS (95% CI)",
        output_dir=RESULT3_FIGURE_DIR,
        stem=f"supplement_result3_full249_expanded_panel_{'cox' if algorithm == 'Cox PH' else 'elasticnet'}_delta_uno",
        color=CNS["purple"], figsize=(8.6, 4.8),
    )
    for horizon in HORIZONS:
        calibration = result3_expanded_calibration.loc[
            result3_expanded_calibration["Algorithm"].eq(algorithm)
            & np.isclose(result3_expanded_calibration["Horizon_Years"], horizon)
        ]
        fig, ax = plt.subplots(figsize=(8.2, 6.0))
        for label, color in [("Clinical + PRS", CNS["black"]), (RESULT3_EXPANDED_LABEL, CNS["purple"])]:
            data = calibration.loc[calibration["Representation"].eq(label)].sort_values("Predicted_Risk")
            ax.plot(data["Predicted_Risk"], data["Observed_KM_Risk"], marker="o", linewidth=2.0, color=color, label=label)
        limit = float(max(0.01, calibration[["Predicted_Risk", "Observed_KM_Risk"]].to_numpy(float).max() * 1.05))
        ax.plot([0, limit], [0, limit], "--", color=CNS["gray"], linewidth=1.2, label="Ideal")
        ax.set_xlim(0, limit); ax.set_ylim(0, limit)
        ax.set_xlabel("Mean predicted CHD risk"); ax.set_ylabel("Kaplan-Meier observed CHD risk")
        ax.set_title(f"Supplementary calibration: {algorithm}, {horizon:g} years", loc="left", fontweight="bold")
        ax.grid(color=CNS["light_gray"], linewidth=0.8); ax.legend(frameon=False)
        clean_axes(ax); fig.tight_layout()
        safe = "cox" if algorithm == "Cox PH" else "elasticnet"
        save_figure(fig, RESULT3_FIGURE_DIR, f"supplement_result3_full249_expanded_panel_{safe}_{int(horizon)}y_calibration")
        plt.show()
        draw_dca(
            result3_expanded_dca.loc[
                result3_expanded_dca["Algorithm"].eq(algorithm)
                & np.isclose(result3_expanded_dca["Horizon_Years"], horizon)
            ],
            title=f"Supplementary survival DCA: {algorithm}, {horizon:g} years",
            output_dir=RESULT3_FIGURE_DIR,
            stem=f"supplement_result3_full249_expanded_panel_{safe}_{int(horizon)}y_dca",
            colors={"Clinical + PRS": CNS["black"], RESULT3_EXPANDED_LABEL: CNS["purple"], "Treat all": CNS["gray"], "Treat none": CNS["gray"]},
        )


In [ ]:
def _pipeline_context(manifest):
    nested = manifest.get("analysis_context", {}) if isinstance(manifest, dict) else {}
    cohort_id = str(manifest.get("cohort_id", nested.get("cohort_id", "unspecified")))
    analysis_role = str(manifest.get("analysis_role", nested.get("analysis_role", "unspecified")))
    if cohort_id.lower() == "coverage90" and analysis_role.lower() == "primary":
        raise RuntimeError("coverage90 must remain a sensitivity cohort; pipeline manifest labels it primary")
    return cohort_id, analysis_role

cohort_id, analysis_role = _pipeline_context(pipeline_manifest)
context_table = pd.DataFrame([{
    "Result": "Result3", "Cohort_ID": cohort_id, "Analysis_Role": analysis_role,
    "Primary_Representations": 3, "Expanded_Panel_Feature_Set": RESULT3_EXPANDED_FEATURE,
    "Expanded_Panel_Role": "expanded_panel_sensitivity",
    "Expanded_Cox_Status": EXPANDED_COX_STATUS,
    "Expanded_Cox_Error": EXPANDED_COX_ERROR,
}])
atomic_write_csv(context_table, os.path.join(RESULT3_SOURCE_DIR, "result3_analysis_context.csv"), index=False)
atomic_write_json({
    "status": "success", "completed_at": str(pd.Timestamp.now()),
    "pipeline_manifest_sha256": current_pipeline_sha256(),
    "cohort_id": cohort_id, "analysis_role": analysis_role,
    "primary_model": "unpenalized Cox PH", "sensitivity_model": "Elastic-net Cox",
    "primary_representation_count": 3,
    "expanded_panel_feature_set": RESULT3_EXPANDED_FEATURE,
    "expanded_panel_role": "expanded_panel_sensitivity",
    "expanded_panel_in_primary_figures": False,
    "expanded_cox_status": EXPANDED_COX_STATUS,
    "expanded_cox_error": EXPANDED_COX_ERROR,
    "expanded_cox_failure_blocks_primary_result3": False,
    "expanded_algorithms_available": RESULT3_EXPANDED_ALGORITHMS,
    "test_used_for_tuning": False,
}, os.path.join(RESULT3_ROOT, "SUCCESS.json"))
